# Augmented clean12 ECG images on Google Colab, saved to Google Drive

The Colab version of `Clean12_Augmented_ECG_Images_EC2.ipynb`. It reads the **clean12 signal dataset**, `ECG_500Hz_clean12.tar.gz`: 113,358 native-500 Hz WFDB records in Account 1's S3. It writes **one augmented ECG sheet (1024x768 PNG) per record**: 113,358 images for 113,358 records. Each image path mirrors its record path (`images/<site>/WFDB/.../<record>.png`), the same layout as the clean12 sheets, so clean and augmented sheets match on `image_path`.

The result goes to **Google Drive**, next to the clean archive, where the clean12 + augmented training notebook reads it:
```text
<DRIVE_DATASET_DIR>/clean_ecg_signal_images_clean12_augmented.tar          ~160 GiB, one tar
<DRIVE_DATASET_DIR>/clean_ecg_signal_images_clean12_augmented.tar.sha256   its SHA-256 (sha256sum format)
```
`DRIVE_DATASET_DIR` defaults to `MyDrive/CardioSentry/new_dataset`. The archive has the same top folder, member paths and metadata files as the EC2 notebook's.

**Per record:**
1. The record is rendered with the **ECG-image-kit**, using the same settings and the same code as `Clean12_ECG_Signal_Images_EC2.ipynb` (both copied from that notebook when this one is built). This gives the 2200x1700 grey sheet the clean12 images were made from.
2. That sheet goes through the image pipeline's **stage-4 paper/photo augmentation**: `stage4_augment.py`, the code behind the `augmented_<grid>_tilt<deg>.png` previews in `correct_prediction_samples`. Its wrinkled-paper texture and fold creases are **stronger** than stage 4 (table below).
3. JPEG q85 round trip, then the clean view's own 1024x768 fit.

The copy of stage 4 in `augment_worker.py` reproduces `stage4_augment.augment` pixel for pixel when run with the stage-4 settings. Only the wrinkle and crease strengths were raised:

| Setting | Stage 4 | This notebook |
|---|---|---|
| wrinkle texture strength | 0.30–0.62 | 0.65–1.00 |
| wrinkle shading limits | ×0.86–1.10 | ×0.76–1.14 |
| crease amplitude | 0.06–0.13 | 0.16–0.26 |
| crease shading limits | ×0.88–1.14 | ×0.76–1.22 |
| crease width (fraction of the sheet's long side) | 0.006–0.020 | 0.004–0.012 (sharper) |
| paper bowing | 0.02–0.06 | 0.03–0.08 |

**Equal representation.** These factors are assigned by plan, not by chance. Grid red is skipped because it is the clean view's grid.

| Factor | Levels | Share each |
|---|---|---|
| grid colour | orange, pink, grey | 1/3 |
| ink strength | 0.72–0.82 (faded), 0.82–0.92, 0.92–1.02 (fresh) | 1/3 |
| tilt | six 1° bins across −3°…+3° | 1/6 |
| perspective | off / on | 1/2 |
| fold creases | off / on | 1/2 |
| edge shadow | off / on | 1/2 |
| blur | off / on | 1/2 |

Every sheet gets wrinkled-paper texture (at the stronger setting above), uneven lighting, colour tint, brightness, contrast, paper-level normalisation and sensor noise, each at a random strength within its range.

How the plan is assigned:
- Levels are dealt round-robin inside each `(site, label set)` stratum, ignoring the labels themselves.
- Each factor uses its own seeded shuffle, so factors are assigned independently of each other.
- Result: level counts are within ±1 overall and within every label combination. §7 writes the audit and stops if it fails.

## How it fits on Colab

| Problem on Colab | What this notebook does |
|---|---|
| ~190 GiB free disk; the PNGs alone are ~160 GiB | Records are rendered in **shards** of ~2,500 (~3.6 GiB). A complete shard is packed into a tar fragment, uploaded to `<archive>.parts/` on Drive and verified, then deleted locally. At most about 4 shards' worth of files are on disk. |
| Sessions end (12-24 h limit, disconnects) | Each finished shard is recorded on Drive. **Run all** in a new runtime downloads the signals again and continues at the first shard that is not on Drive. A lost session costs only the shards that were still rendering. |
| Drive files cannot be appended to | After the last shard, **one resumable Drive upload** streams the fragments back from Drive, followed by the metadata files and the tar end marker, into the single archive. That join resumes across sessions too. Its whole-file SHA-256 is computed while the shards are made, carried from session to session, and checked against Drive's own `sha256Checksum`. |
| CPU count varies by runtime | §1 detects the usable vCPUs, RAM and disk; one render worker per vCPU, capped by RAM. |

**The GPU is not used.** The ECG-image-kit and the augmentation run on the CPU only, so what matters is the number of vCPUs, which §1 prints. On an L4 runtime the GPU sits idle; if a CPU-only runtime offers as many vCPUs, it does the same work for fewer compute units.

**Cost.** Writing to and reading from Drive is free within your storage quota. The only AWS charge is the signal download, ~5.5 GB of S3 egress from Account 1 per new runtime (on the order of USD 0.50 each).

## Before you run (once)

**1. Drive space.** The account you sign in with at the Drive prompts needs free storage for the fragments plus the final archive: about **320 GiB** during the join. With less (but at least ~165 GiB) the join deletes each fragment as soon as it has been copied, see §11. The preflight (§9) checks the quota. Files this notebook creates count against that account, even inside a folder shared by another account.

**2. Kit bundle.** Two files from `prototype-code/dataset_generation_code/` in the repo: `ecg_image_v1_code.tar.gz` and `ecg_image_v1_code.tar.gz.sha256`. It is the ECG-image-kit (code plus the wrinkled-paper photographs), 12 MB. Use one of these:
- **Recommended:** put both files in Drive at `MyDrive/CardioSentry/codes/clean12_augmented_ecg_images_colab/` (`KIT_DRIVE_DIR`). §4 copies them in every new runtime.
- Upload both to `/content` with the Files pane (gone when the runtime is recycled).
- Put both next to the signals in Account 1 (`S3_KIT_SOURCE`); §4 downloads them if no other copy is found.

**3. AWS read key** in Colab **Secrets** (key icon; turn on notebook access for each): `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, optionally `AWS_DEFAULT_REGION` (default `us-east-1`). The key needs only:
```json
{"Effect": "Allow", "Action": "s3:GetObject",
 "Resource": "arn:aws:s3:::cardiosentry-fcmae-758821198776-us-east-1-an/CardioSentry/dataset/clean12/*"}
```
An Account 1 key with that policy needs nothing else. A key from another account also needs the same statement in Account 1's bucket policy, with that user as `Principal`.

**4. Runtime.** Runtime > Change runtime type: pick the runtime with the **most vCPUs** you can get (High-RAM if offered). Then **Runtime > Run all**. Sign in at the two Google prompts (Drive mount, then Drive API) **with the same account**, and keep the tab open. A session that ends is fine: reconnect and Run all again.

**Run state on Drive:** `<DRIVE_DATASET_DIR>/clean_ecg_signal_images_clean12_augmented.parts/`. It holds:
- `run_config.json`;
- per shard: `shard_NNNNN_of_NNNNN.tarpart`, `.manifest.csv` and `.json`;
- during the join: `tail.bin` and `join.json`;
- at the end: `compose.json`.

Fragments and the join files are deleted once the archive is verified; the small state files stay.

In [ ]:
# 1. Configuration and runtime hardware: edit before running any later cell.
import os
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"

import shutil
import subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Account 1 (read-only, cross-account): the clean12 signal dataset.
S3_SIGNAL_PREFIX = "s3://cardiosentry-fcmae-758821198776-us-east-1-an/CardioSentry/dataset/clean12/"
S3_SOURCE = S3_SIGNAL_PREFIX + "ECG_500Hz_clean12.tar.gz"
S3_SHA256_SOURCE = S3_SIGNAL_PREFIX + "ECG_500Hz_clean12.tar.gz.sha256"
# Fallback only: the kit bundle normally comes from /content or KIT_DRIVE_DIR.
S3_KIT_SOURCE = S3_SIGNAL_PREFIX + "ecg_image_v1_code.tar.gz"
S3_KIT_SHA256_SOURCE = S3_SIGNAL_PREFIX + "ecg_image_v1_code.tar.gz.sha256"
KIT_DRIVE_DIR = "/content/drive/MyDrive/CardioSentry/codes/clean12_augmented_ecg_images_colab"

# Google Drive: the folder that receives <basename>.tar (+ .sha256) and the run state
# <basename>.parts/. new_dataset/ is where the clean12 + augmented training notebook
# looks. DRIVE_DATASET_FOLDER_ID (the id in the folder's URL) overrides the path lookup.
DRIVE_DATASET_DIR = "/content/drive/MyDrive/CardioSentry/new_dataset"
DRIVE_DATASET_FOLDER_ID = ""
ARCHIVE_BASENAME = "clean_ecg_signal_images_clean12_augmented"
DRIVE_API = "https://www.googleapis.com"
DRIVE_CHUNK_BYTES = 64 * 2**20   # resumable-upload chunk; a multiple of 256 KiB

WORK_DIR = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ecg_clean12_augment"
LOCAL_ARCHIVE = WORK_DIR / "input" / "ECG_500Hz_clean12.tar.gz"
INPUT_DIR = WORK_DIR / "input" / "ECG_500Hz_clean12"
OUTPUT_DIR = WORK_DIR / ARCHIVE_BASENAME   # metadata files; the archive's top folder
SHARDS_DIR = WORK_DIR / "shards"           # images of the shards not yet on Drive

EXPECTED_RECORDS = 113_358
CONDITIONS = ["SB", "STACH", "LBBB", "LQT", "AF", "AFL",
              "3AVB", "2AVB", "1AVB", "STEMI", "STE", "NORM"]
# Kit/render settings: copied from Clean12_ECG_Signal_Images_EC2.ipynb at build
# time, so every grey master is the master its clean12 sheet was made from.
EXPECTED_FS = 500
EXPECTED_SAMPLES = 5000
MAX_DURATION_S = 10.0
KIT_SEED = 20260918
KIT_FIXED_FONT = 'Arial.ttf'
KIT_RENDER_FLAGS = ['--num_columns', '4', '--full_mode', 'II', '-r', '200', '--pad_inches', '0', '--calibration_pulse', '1', '--random_bw', '1']
KIT_ENV = {'ECG_ROW_GAP': '4.4', 'ECG_RHYTHM_GAP': '5.5', 'ECG_BOTTOM_RESERVE': '1.9'}
PNG_COMPRESS_LEVEL = 3
# The kit's lead-label glyphs depend on the matplotlib version; 3.11.1 reproduces
# the clean12 sheets pixel for pixel. §2 installs it, §9 checks it.
MATPLOTLIB_VERSION = "3.11.1"
AUGMENT_SEED = 20261023          # plan shuffles + per-record effect draws
EST_IMAGE_MIB = 1.5              # measured 1.13-1.49 MiB per augmented sheet

# Shards: ~2,500 records (~3.6 GiB) each. Fixed for a run: the layout is locked on Drive (§8).
SHARD_RECORDS = 2_500
SHARD_LOOKAHEAD = 2              # shards rendered ahead of the one uploading

# Release the Colab runtime (stop using compute units) once the archive is verified
# on Drive (§12), and only if every pipeline cell completed and no cell raised.
RELEASE_RUNTIME_WHEN_DONE = True
RELEASE_GRACE_SECONDS = 60       # interrupt the kernel within this window to cancel


def detect_cpus():
    # vCPUs this process may use: CPU affinity, capped by a cgroup CPU quota.
    cpus = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
    for quota_file, period_file in (("/sys/fs/cgroup/cpu.max", None),
                                    ("/sys/fs/cgroup/cpu/cpu.cfs_quota_us", "/sys/fs/cgroup/cpu/cpu.cfs_period_us")):
        try:
            if period_file is None:
                quota, period = Path(quota_file).read_text().split()[:2]
            else:
                quota, period = Path(quota_file).read_text().strip(), Path(period_file).read_text().strip()
        except (OSError, ValueError):
            continue
        if quota not in ("max", "-1"):
            cpus = min(cpus, max(1, int(quota) // int(period)))
        break
    return cpus


def detect_ram_gib():
    total = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
    for path in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
        try:
            value = Path(path).read_text().strip()
        except OSError:
            continue
        if value.isdigit():
            total = min(total, int(value))
    return total / 2**30


def describe_cpu():
    model, cores, socket = "unknown", set(), None
    try:
        for line in Path("/proc/cpuinfo").read_text().splitlines():
            key, _, value = (part.strip() for part in line.partition(":"))
            if key == "model name":
                model = value
            elif key == "physical id":
                socket = value
            elif key == "core id":
                cores.add((socket, value))
    except OSError:
        pass
    return model, len(cores) or None


USABLE_CPUS = detect_cpus()
RAM_GIB = detect_ram_gib()
CPU_MODEL, PHYSICAL_CORES = describe_cpu()
GPU_NAME = None
if shutil.which("nvidia-smi"):
    _gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True)
    if _gpu.returncode == 0 and _gpu.stdout.strip():
        GPU_NAME = _gpu.stdout.strip().splitlines()[0]

# One render worker per usable vCPU, capped by RAM (~0.7 GiB per worker + 3 GiB).
# A worker runs one kit process per batch, then augments that batch's masters.
WORKERS = max(1, min(USABLE_CPUS, int((RAM_GIB - 3) / 0.7)))
BATCH_SIZE = 16                  # small batches keep every worker busy at shard ends
MAX_PENDING_BATCHES = 3 * WORKERS   # queued work keeps workers busy while a shard uploads
PROGRESS_EVERY = 500
RETRY_PASSES = 1                 # a failed record is retried once before its shard fails
_shm = Path("/dev/shm")
KIT_SCRATCH_DIR = (_shm / "ecg_clean12_augment_scratch"
                   if os.access(_shm, os.W_OK) and shutil.disk_usage(_shm).free >= WORKERS * 256 * 2**20
                   else WORK_DIR / "kit_scratch")
S3_TRANSFER = dict(max_concurrency=32, multipart_chunksize=64 * 2**20, use_threads=True)

# Safety record for §12. Re-running THIS cell starts a fresh record: both reset.
STAGES_DONE = set()              # each pipeline cell adds its name when it completes
CELL_ERRORS = []                 # every cell that raised in this kernel since this cell
try:
    _ipython = get_ipython()
except NameError:                # plain Python (no IPython): only STAGES_DONE applies
    _ipython = None
if _ipython is not None:
    for _callback in list(_ipython.events.callbacks["post_run_cell"]):
        if getattr(_callback, "__name__", "") == "_record_cell_error":
            _ipython.events.unregister("post_run_cell", _callback)

    def _record_cell_error(result):
        if not result.success:
            error = result.error_in_exec or result.error_before_exec
            CELL_ERRORS.append(f"{type(error).__name__}: {error}"[:300])

    _ipython.events.register("post_run_cell", _record_cell_error)

assert WORKERS >= 1 and BATCH_SIZE >= 1 and MAX_PENDING_BATCHES >= WORKERS and SHARD_LOOKAHEAD >= 0
assert DRIVE_CHUNK_BYTES % (256 * 1024) == 0
WORK_DIR.mkdir(parents=True, exist_ok=True)
SHARDS_DIR.mkdir(parents=True, exist_ok=True)
_disk = shutil.disk_usage(WORK_DIR)
print(f"Runtime: {'Colab' if IN_COLAB else 'not Colab'} | CPU {CPU_MODEL}")
print(f"  usable vCPUs {USABLE_CPUS}" + (f" ({PHYSICAL_CORES} physical cores)" if PHYSICAL_CORES else "")
      + f" | RAM {RAM_GIB:.1f} GiB | disk {_disk.free / 2**30:.0f} GiB free of {_disk.total / 2**30:.0f} GiB"
      + f" | GPU {GPU_NAME or 'none'}" + (" (not used)" if GPU_NAME else ""))
print(f"  -> {WORKERS} render workers, batches of {BATCH_SIZE}; kit scratch {KIT_SCRATCH_DIR}")
print("Working directory:", WORK_DIR)
print("Source:", S3_SOURCE)
print("Archive:", f"{DRIVE_DATASET_DIR}/{ARCHIVE_BASENAME}.tar", "| release runtime when done:", RELEASE_RUNTIME_WHEN_DONE)

## 2. Install and verify runtime dependencies

The clean12 render notebook's install cell, unchanged, after one Colab addition: matplotlib is pinned to `MATPLOTLIB_VERSION`, because Colab ships an older matplotlib and the kit's lead-label glyphs change between versions. The kit runs as a subprocess of this kernel's Python, so the pinned version is the one it uses even if the kernel had already imported another. `google-auth` and `requests` (for the Drive API) are already on Colab.

In [ ]:
%pip install --quiet "matplotlib=={MATPLOTLIB_VERSION}" google-auth requests
# Install ALL runtime packages into this kernel's environment BEFORE any
# third-party import. Unconditional (a fast no-op when already satisfied) so a
# half-present environment can never slip through a presence check and fail
# later at `import scipy` with ModuleNotFoundError.
%pip install --quiet boto3 numpy scipy Pillow matplotlib pandas qrcode pyyaml wfdb
import importlib as _package_importlib
import subprocess as _package_subprocess
import sys as _kernel_sys

# boto3 = S3 I/O; numpy/scipy/Pillow = worker staging + recolour + resize;
# matplotlib/pandas/qrcode/pyyaml/wfdb = the kit batch-driver imports.
_module_to_spec = {"boto3": "boto3", "numpy": "numpy", "scipy": "scipy",
                   "PIL": "Pillow", "matplotlib": "matplotlib",
                   "pandas": "pandas", "qrcode": "qrcode",
                   "yaml": "pyyaml", "wfdb": "wfdb>=4.3"}

# wfdb>=4.3 is REQUIRED (4.1.x dies at import on pandas 3): upgrade a stale
# copy BEFORE it is ever imported in this process (upgrading after a failed
# import mixes versions and needs a kernel restart instead).
try:
    from importlib.metadata import version as _dist_version
    _wfdb_have = _dist_version("wfdb")
    _wfdb_old = tuple(int(p) for p in _wfdb_have.split(".")[:2]) < (4, 3)
except Exception:
    _wfdb_have, _wfdb_old = "?", False
if _wfdb_old:
    if "wfdb" in _kernel_sys.modules:
        raise RuntimeError(
            f"wfdb {_wfdb_have} is already imported and too old (need >=4.3): "
            "restart the kernel and re-run from §1.")
    print(f"Upgrading stale wfdb {_wfdb_have} to wfdb>=4.3 ...", flush=True)
    _package_subprocess.check_call(
        [_kernel_sys.executable, "-m", "pip", "install", "--quiet",
         "--retries", "5", "--timeout", "300", "wfdb>=4.3"])
    _package_importlib.invalidate_caches()

# Verify every import; an unimportable module gets one automatic repair
# install (covers detection gaps and broken copies) instead of failing later.
for _mod, _spec in _module_to_spec.items():
    try:
        _package_importlib.import_module(_mod)
    except Exception as exc:
        print(f"{_mod} not importable ({type(exc).__name__}); "
              f"repair-installing {_spec} ...", flush=True)
        _package_subprocess.check_call(
            [_kernel_sys.executable, "-m", "pip", "install", "--quiet",
             "--retries", "5", "--timeout", "300", _spec])
        _package_importlib.invalidate_caches()
        _package_importlib.import_module(_mod)
print("Runtime packages verified:",
      ", ".join(sorted(_module_to_spec)), flush=True)

import csv
from collections import Counter
import hashlib
import importlib
import json
import math
import multiprocessing as mp
import shutil
import sys
import tarfile
import time
from concurrent.futures import FIRST_COMPLETED, ProcessPoolExecutor, wait
from datetime import datetime, timezone
from urllib.parse import urlparse

import boto3
import numpy as np
import scipy
from scipy.io import loadmat as _checked_loadmat
from PIL import Image
from boto3.s3.transfer import TransferConfig

print("Python", sys.version.split()[0], "| boto3", boto3.__version__,
      "| NumPy", np.__version__, "| SciPy", scipy.__version__,
      "| Pillow", Image.__version__)
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import subprocess
import urllib.request
from collections import defaultdict
from pathlib import PurePosixPath

import pandas as pd
from botocore.exceptions import ClientError
STAGES_DONE.add("install")

## 3. Google Drive and AWS access

- **Drive:** mounts Drive (prompt 1), then signs in to the Drive API (prompt 2, **same account**). The archive and the run state are written through the API, which is authoritative and resumable; the mount is used only to read the kit and folder ids.
- **Destination folder:** resolves `DRIVE_DATASET_DIR` to its Drive id. The id comes from the mount's `user.drive.id` attribute, or else by walking the path with the API and following shortcuts. `DRIVE_DATASET_FOLDER_ID` overrides both.
- **Account check:** prints the signed-in account and its free storage.
- **AWS:** copies the read key from Colab Secrets and prints its identity, so a wrong key shows up here rather than hours later.

In [ ]:
import time
import requests

FOLDER_MIME = "application/vnd.google-apps.folder"
SHORTCUT_MIME = "application/vnd.google-apps.shortcut"
DRIVE_FILE_FIELDS = "id,name,mimeType,size,md5Checksum,sha256Checksum,shortcutDetails,capabilities/canAddChildren"


class DriveTransientError(Exception):
    pass


class DriveSessionExpired(Exception):
    pass


def drive_authorized_session():
    import google.auth
    from google.auth.transport.requests import AuthorizedSession
    if IN_COLAB:
        from google.colab import auth as colab_auth
        colab_auth.authenticate_user()
    credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
    return AuthorizedSession(credentials)


def drive_request(method, path, ok=(200,), tries=8, **kwargs):
    # One Drive API call; retries rate limits, server errors and dropped connections.
    url = path if path.startswith("http") else DRIVE_API + path
    error = None
    for attempt in range(tries):
        try:
            response = drive_http.request(method, url, timeout=(30, 600), **kwargs)
        except requests.RequestException as exc:
            error = f"{type(exc).__name__}: {exc}"
        else:
            if response.status_code in ok:
                return response
            retryable = response.status_code in (408, 429, 500, 502, 503, 504) or (
                response.status_code == 403 and "ateLimitExceeded" in response.text)
            if not retryable:
                raise RuntimeError(f"Drive API {method} {url.split('?')[0]} -> {response.status_code}: "
                                   f"{response.text[:500]}")
            error = f"{response.status_code}: {response.text[:200]}"
        time.sleep(min(60, 2 ** attempt))
    raise RuntimeError(f"Drive API {method} {url.split('?')[0]} failed {tries} times; last: {error}")


def drive_children(folder_id, name=None):
    query = f"'{folder_id}' in parents and trashed=false"
    if name is not None:
        query += " and name='" + name.replace("\\", "\\\\").replace("'", "\\'") + "'"
    files, token = [], None
    while True:
        params = {"q": query, "fields": f"nextPageToken,files({DRIVE_FILE_FIELDS})", "pageSize": 1000,
                  "supportsAllDrives": "true", "includeItemsFromAllDrives": "true"}
        if token:
            params["pageToken"] = token
        page = drive_request("GET", "/drive/v3/files", params=params).json()
        files += page.get("files", [])
        token = page.get("nextPageToken")
        if not token:
            return files


def drive_meta(file_id):
    response = drive_request("GET", f"/drive/v3/files/{file_id}", ok=(200, 404),
                             params={"fields": DRIVE_FILE_FIELDS, "supportsAllDrives": "true"})
    return response.json() if response.status_code == 200 else None


def drive_download(file_id, start=None, length=None):
    headers = {"Range": f"bytes={start}-{start + length - 1}"} if start is not None else {}
    data = drive_request("GET", f"/drive/v3/files/{file_id}", ok=(200, 206), headers=headers,
                         params={"alt": "media", "supportsAllDrives": "true"}).content
    if length is not None and len(data) != length:
        raise DriveTransientError(f"ranged download returned {len(data)} of {length} bytes")
    return data


def drive_delete(file_id):
    drive_request("DELETE", f"/drive/v3/files/{file_id}", ok=(200, 204, 404),
                  params={"supportsAllDrives": "true"})


def drive_folder(parent_id, name):
    # The one folder called name in parent_id, created if missing.
    hits = [f for f in drive_children(parent_id, name) if f["mimeType"] == FOLDER_MIME]
    if len(hits) > 1:
        raise RuntimeError(f"{len(hits)} folders named {name!r} in Drive folder {parent_id}; keep one")
    if hits:
        return hits[0]["id"]
    return drive_request("POST", "/drive/v3/files", params={"supportsAllDrives": "true", "fields": "id"},
                         json={"name": name, "mimeType": FOLDER_MIME, "parents": [parent_id]}).json()["id"]


def drive_free_bytes():
    quota = drive_request("GET", "/drive/v3/about", params={"fields": "storageQuota"}).json()["storageQuota"]
    return None if "limit" not in quota else int(quota["limit"]) - int(quota["usage"])


def drive_resumable_start(parent_id, name, size=None):
    # A resumable upload session (valid for a week); size=None leaves the length open.
    headers = {"X-Upload-Content-Type": "application/octet-stream"}
    if size is not None:
        headers["X-Upload-Content-Length"] = str(size)
    response = drive_request("POST", "/upload/drive/v3/files", headers=headers,
                             params={"uploadType": "resumable", "supportsAllDrives": "true",
                                     "fields": DRIVE_FILE_FIELDS},
                             json={"name": name, "parents": [parent_id]})
    return response.headers["Location"]


def _committed(response):
    # Bytes Drive holds after a 308: "Range: bytes=0-N" means N + 1; no header means 0.
    value = response.headers.get("Range")
    return int(value.rsplit("-", 1)[1]) + 1 if value else 0


def drive_resumable_status(uri, total, declared=True):
    # (bytes Drive holds, file metadata if the upload is already complete).
    for attempt in range(12):
        try:
            response = drive_http.put(uri, data=b"", timeout=(30, 120),
                                      headers={"Content-Range": f"bytes */{total if declared else '*'}"})
        except requests.RequestException:
            time.sleep(min(60, 2 ** attempt))
            continue
        if response.status_code in (200, 201):
            return total, response.json()
        if response.status_code == 308:
            return _committed(response), None
        if response.status_code in (404, 410):
            raise DriveSessionExpired(f"upload session expired ({response.status_code})")
        if response.status_code not in (408, 429, 500, 502, 503, 504):
            raise RuntimeError(f"upload status -> {response.status_code}: {response.text[:300]}")
        time.sleep(min(60, 2 ** attempt))
    raise RuntimeError("upload status query kept failing")


def drive_resumable_upload(uri, total, read, declared=True, on_commit=None):
    # Send bytes [committed, total) in chunks; read(offset, n) supplies them. Returns file metadata.
    committed, meta = drive_resumable_status(uri, total, declared)
    failures = 0
    while meta is None:
        if on_commit:
            on_commit(committed)
        start = committed
        data = read(start, min(DRIVE_CHUNK_BYTES, total - start))
        end = start + len(data)
        size_field = total if declared or end == total else "*"
        try:
            response = drive_http.put(uri, data=data, timeout=(30, 900),
                                      headers={"Content-Range": f"bytes {start}-{end - 1}/{size_field}"})
            if response.status_code in (200, 201):
                committed, meta = total, response.json()
            elif response.status_code == 308:
                committed, failures = _committed(response), 0
            elif response.status_code in (404, 410):
                raise DriveSessionExpired(f"upload session expired ({response.status_code})")
            elif response.status_code in (408, 429, 500, 502, 503, 504):
                raise DriveTransientError(f"{response.status_code}: {response.text[:200]}")
            else:
                raise RuntimeError(f"upload chunk -> {response.status_code}: {response.text[:300]}")
        except (requests.RequestException, DriveTransientError) as exc:
            failures += 1
            if failures > 12:
                raise RuntimeError(f"upload kept failing at byte {start:,}: {exc}") from exc
            time.sleep(min(60, 2 ** failures))
            committed, meta = drive_resumable_status(uri, total, declared)
    if on_commit:
        on_commit(total)
    return meta


def check_drive_copy(meta, size, sha256_hex, md5_hex=None):
    # Drive's own checksums must equal the local ones.
    if int(meta.get("size", -1)) != size:
        raise RuntimeError(f"Drive file {meta.get('name')} is {meta.get('size')} bytes, expected {size:,}")
    if meta.get("sha256Checksum"):
        if meta["sha256Checksum"] != sha256_hex:
            raise RuntimeError(f"Drive file {meta['name']} sha256 {meta['sha256Checksum']} != local {sha256_hex}")
    elif md5_hex and meta.get("md5Checksum"):
        if meta["md5Checksum"] != md5_hex:
            raise RuntimeError(f"Drive file {meta['name']} md5 {meta['md5Checksum']} != local {md5_hex}")
    else:
        raise RuntimeError(f"Drive returned no checksum for {meta.get('name')}")


def drive_put_bytes(parent_id, name, body):
    # Replace any same-named files in parent_id with one file holding body; verified.
    for stale in drive_children(parent_id, name):
        drive_delete(stale["id"])
    uri = drive_resumable_start(parent_id, name, len(body))
    meta = drive_resumable_upload(uri, len(body), lambda offset, n: body[offset:offset + n])
    check_drive_copy(meta, len(body), hashlib.sha256(body).hexdigest(), hashlib.md5(body).hexdigest())
    return meta


def drive_get_bytes(parent_id, name):
    hits = drive_children(parent_id, name)
    if len(hits) > 1:
        raise RuntimeError(f"{len(hits)} files named {name!r} in Drive folder {parent_id}")
    return drive_download(hits[0]["id"]) if hits else None


def resolve_drive_folder(mounted_path, folder_id=""):
    # Drive id of a mounted folder: explicit id, the mount's xattr, or an API walk from My Drive.
    if folder_id:
        return folder_id, "DRIVE_DATASET_FOLDER_ID"
    try:
        return os.getxattr(mounted_path, "user.drive.id").decode().strip(), "mount attribute"
    except (OSError, AttributeError):
        pass
    parts = Path(mounted_path).relative_to("/content/drive/MyDrive").parts
    parent = "root"
    for part in parts:
        hits = [f for f in drive_children(parent, part) if f["mimeType"] in (FOLDER_MIME, SHORTCUT_MIME)]
        if len(hits) != 1:
            raise RuntimeError(f"{len(hits)} folders named {part!r} on the way to {mounted_path}; create the "
                               "folder (or set DRIVE_DATASET_FOLDER_ID to its id from the folder's URL)")
        parent = hits[0]["shortcutDetails"]["targetId"] if hits[0]["mimeType"] == SHORTCUT_MIME else hits[0]["id"]
    return parent, "Drive API path walk"


if IN_COLAB and not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
drive_http = drive_authorized_session()
_about = drive_request("GET", "/drive/v3/about", params={"fields": "user(emailAddress)"}).json()
dataset_folder_id, _how = resolve_drive_folder(DRIVE_DATASET_DIR, DRIVE_DATASET_FOLDER_ID)
_folder = drive_meta(dataset_folder_id)
if not _folder or _folder["mimeType"] != FOLDER_MIME:
    raise RuntimeError(f"Drive id {dataset_folder_id} ({_how}) is not a folder this account can see")
if not _folder.get("capabilities", {}).get("canAddChildren", False):
    raise RuntimeError(f"Account {_about['user']['emailAddress']} cannot add files to Drive folder {_folder['name']!r}")
_free = drive_free_bytes()
print(f"Google Drive: {_about['user']['emailAddress']} | destination {DRIVE_DATASET_DIR} = folder "
      f"{_folder['name']!r} ({dataset_folder_id}, via {_how}) | free storage "
      + ("unlimited" if _free is None else f"{_free / 2**30:,.0f} GiB"))


def colab_secret(name):
    # An environment variable wins; otherwise the Colab secret of that name.
    if os.environ.get(name):
        return os.environ[name].strip()
    if not IN_COLAB:
        return None
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:  # secret absent, or notebook access not granted
        return None
    return value.strip() if value else None


for _name in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"):
    _value = colab_secret(_name)
    if _value:
        os.environ[_name] = _value
os.environ["AWS_DEFAULT_REGION"] = colab_secret("AWS_DEFAULT_REGION") or "us-east-1"
if boto3.Session().get_credentials() is None:
    raise RuntimeError("No AWS credentials. In Colab open Secrets (key icon), add AWS_ACCESS_KEY_ID and "
                       "AWS_SECRET_ACCESS_KEY, turn on notebook access for both, and rerun this cell.")
try:
    _arn = boto3.Session().client("sts").get_caller_identity()["Arn"]
except Exception as exc:
    raise RuntimeError(f"The AWS key was rejected: {type(exc).__name__}: {exc}") from exc
print(f"AWS read key (Account 1 signals): {_arn}")
STAGES_DONE.add("access")

## 3b. Download, verify and extract the signals (Account 1)

The clean12 render notebook's download cell, unchanged:
- it reads the `.sha256` sidecar, downloads the archive (5.5 GiB) and hashes all of it;
- it extracts safely, rejecting links and any path outside the destination;
- a completion marker lets a rerun in the same runtime skip the work.

A new runtime downloads it again (Colab's disk is wiped). `AccessDenied` here means the read key lacks `s3:GetObject` on the signals (top cell).

In [ ]:
def split_s3_uri(uri):
    parsed = urlparse(uri)
    if parsed.scheme != "s3" or not parsed.netloc or ":" in parsed.netloc or not parsed.path.strip("/"):
        raise ValueError(f"Expected s3://bucket/key, got {uri!r}")
    return parsed.netloc, parsed.path.lstrip("/")

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while True:
            block = stream.read(8 * 1024 * 1024)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

from botocore.exceptions import NoCredentialsError, PartialCredentialsError

aws_session = boto3.Session()
try:
    aws_credentials = aws_session.get_credentials()
except (NoCredentialsError, PartialCredentialsError) as exc:
    aws_credentials = None
if aws_credentials is None:
    raise RuntimeError(
        "No AWS credentials are available to this Jupyter kernel. Attach an EC2 IAM "
        "role to this instance: EC2 console > Instances > select instance > Actions > "
        "Security > Modify IAM role. Grant read access to the dataset archive and "
        "checksum, plus write/read access under the S3_UPLOAD_PREFIX. Then rerun this cell."
    )
print("AWS credential source:", aws_credentials.method)
s3 = aws_session.client("s3")
src_bucket, src_key = split_s3_uri(S3_SOURCE)
sha_bucket, sha_key = split_s3_uri(S3_SHA256_SOURCE)
try:
    checksum_object = s3.get_object(Bucket=sha_bucket, Key=sha_key)
except (NoCredentialsError, PartialCredentialsError) as exc:
    raise RuntimeError(
        "AWS credentials could not be used. Check the EC2 instance IAM role and "
        "instance metadata access, then rerun this cell."
    ) from exc
if int(checksum_object["ContentLength"]) > 1024:
    raise ValueError("S3 SHA-256 sidecar is unexpectedly large")
checksum_fields = checksum_object["Body"].read().decode("utf-8").strip().split()
if len(checksum_fields) != 2 or checksum_fields[1].lstrip("*") != LOCAL_ARCHIVE.name:
    raise ValueError("SHA-256 sidecar must name the dataset archive")
expected_sha256 = checksum_fields[0].lower()
if len(expected_sha256) != 64 or any(char not in "0123456789abcdef" for char in expected_sha256):
    raise ValueError("SHA-256 sidecar does not contain a valid digest")

src_info = s3.head_object(Bucket=src_bucket, Key=src_key)
source_bytes = int(src_info["ContentLength"])
source_identity = {"uri": S3_SOURCE, "sha256_uri": S3_SHA256_SOURCE,
                   "sha256": expected_sha256, "bytes": source_bytes,
                   "etag": src_info.get("ETag", "")}
source_sidecar = LOCAL_ARCHIVE.with_name(LOCAL_ARCHIVE.name + ".source.json")
archive_matches = (LOCAL_ARCHIVE.is_file() and LOCAL_ARCHIVE.stat().st_size == source_bytes
                   and file_sha256(LOCAL_ARCHIVE) == expected_sha256)
if archive_matches:
    print("Reusing SHA-256 verified source archive:", LOCAL_ARCHIVE)
else:
    partial = LOCAL_ARCHIVE.with_name(LOCAL_ARCHIVE.name + ".part")
    partial.unlink(missing_ok=True)
    print(f"Downloading {S3_SOURCE} ({source_bytes / 2**30:.2f} GiB) ...")
    s3.download_file(src_bucket, src_key, str(partial),
                     Config=TransferConfig(**S3_TRANSFER))
    if partial.stat().st_size != source_bytes:
        raise IOError("Downloaded byte count differs from S3 ContentLength")
    if file_sha256(partial) != expected_sha256:
        partial.unlink(missing_ok=True)
        raise IOError("Downloaded archive SHA-256 does not match S3 sidecar")
    partial.replace(LOCAL_ARCHIVE)
    print("Downloaded and SHA-256 verified:", LOCAL_ARCHIVE)
source_sidecar.write_text(json.dumps(source_identity, indent=2) + "\n")

extraction_root = INPUT_DIR.parent.resolve()
marker = extraction_root / ".ecg_clean12_extraction_complete.json"
marker_value = {"source_uri": S3_SOURCE, "archive_sha256": expected_sha256,
                "archive_bytes": source_bytes,
                "archive_mtime": LOCAL_ARCHIVE.stat().st_mtime_ns}
already_extracted = False
if marker.is_file():
    try:
        already_extracted = json.loads(marker.read_text()) == marker_value and INPUT_DIR.is_dir()
    except (OSError, ValueError):
        pass
if already_extracted:
    print("Reusing extracted clean12 ECG tree:", INPUT_DIR)
else:
    marker.unlink(missing_ok=True)
    print("Extracting to", extraction_root, "...")
    member_count = 0
    from pathlib import PurePosixPath
    with tarfile.open(LOCAL_ARCHIVE, "r:gz") as archive:
        for member in archive:
            rel = PurePosixPath(member.name)
            if rel.is_absolute() or ".." in rel.parts or not rel.parts or rel.parts[0] != INPUT_DIR.name:
                raise ValueError(f"Unsafe/unexpected archive path: {member.name!r}")
            dest = extraction_root.joinpath(*rel.parts)
            if not dest.resolve(strict=False).is_relative_to(extraction_root):
                raise ValueError(f"Archive path resolves outside destination: {member.name!r}")
            if member.isdir():
                dest.mkdir(parents=True, exist_ok=True)
            elif member.isfile():
                if dest.is_symlink():
                    raise ValueError(f"Refusing to overwrite symlink: {dest}")
                dest.parent.mkdir(parents=True, exist_ok=True)
                stream = archive.extractfile(member)
                if stream is None:
                    raise IOError(f"Cannot read archive member {member.name}")
                with stream, dest.open("wb") as out:
                    shutil.copyfileobj(stream, out, length=1024 * 1024)
                if dest.stat().st_size != member.size:
                    raise IOError(f"Incomplete extracted member {member.name}")
                member_count += 1
            else:
                raise ValueError(f"Links/special files are disallowed: {member.name!r}")
    if not INPUT_DIR.is_dir():
        raise FileNotFoundError(f"Archive did not create {INPUT_DIR}")
    marker.write_text(json.dumps(marker_value, indent=2) + "\n")
    print(f"Extracted {member_count:,} files")

STAGES_DONE.add("download")

## 4. Kit bundle: locate, verify, extract, patch gate

Colab addition first: if the two kit files are not in `/content`, they are copied from `KIT_DRIVE_DIR`. Then the clean12 render notebook's kit cell, unchanged: it falls back to `S3_KIT_SOURCE` if there is still no local copy, the bundle must match its `.sha256` sidecar, and rendering is refused unless every kit patch marker and the fixed font are present. The wrinkled-paper photographs come from the same bundle.

In [ ]:
_kit_files = ("ecg_image_v1_code.tar.gz", "ecg_image_v1_code.tar.gz.sha256")
if KIT_DRIVE_DIR and not all((Path.cwd() / name).is_file() for name in _kit_files):
    if all((Path(KIT_DRIVE_DIR) / name).is_file() for name in _kit_files):
        for name in _kit_files:
            shutil.copy2(Path(KIT_DRIVE_DIR) / name, Path.cwd() / name)
        print("Kit bundle copied from", KIT_DRIVE_DIR)
    else:
        print(f"Kit bundle not in {Path.cwd()} or {KIT_DRIVE_DIR}; trying the other locations")

KIT_BUNDLE = None
_kit_search = [Path.cwd() / "ecg_image_v1_code.tar.gz",
               WORK_DIR / "bundle" / "ecg_image_v1_code.tar.gz",
               WORK_DIR / "ecg_image_v1_code.tar.gz"]
for _ancestor in [Path.cwd(), *list(Path.cwd().parents)[:4]]:
    _kit_search.append(_ancestor / "dataset_generation_code" / "ecg_image_v1_code.tar.gz")
for _cand in _kit_search:
    if _cand.is_file():
        KIT_BUNDLE = _cand
        break
if KIT_BUNDLE is None:
    # No local copy — pull the kit bundle from S3 with the same sidecar-pinned
    # discipline as the source archive, then fall through to the checks below.
    try:
        print("No local kit bundle; downloading from", S3_KIT_SOURCE, "...", flush=True)
        _kit_dest = WORK_DIR / "bundle" / "ecg_image_v1_code.tar.gz"
        _kit_dest.parent.mkdir(parents=True, exist_ok=True)
        _kb, _kk = split_s3_uri(S3_KIT_SOURCE)
        _sb, _sk = split_s3_uri(S3_KIT_SHA256_SOURCE)
        _sobj = s3.get_object(Bucket=_sb, Key=_sk)
        if int(_sobj["ContentLength"]) > 1024:
            raise ValueError("Kit SHA-256 sidecar is unexpectedly large")
        _raw = _sobj["Body"].read()
        (_kit_dest.parent / (_kit_dest.name + ".sha256")).write_bytes(_raw)
        _fields = _raw.decode("utf-8").strip().split()
        if len(_fields) != 2 or _fields[1].lstrip("*") != _kit_dest.name:
            raise ValueError("Kit SHA-256 sidecar must name the kit bundle")
        _want = _fields[0].lower()
        if len(_want) != 64 or any(c not in "0123456789abcdef" for c in _want):
            raise ValueError("Kit SHA-256 sidecar does not contain a valid digest")
        _kinfo = s3.head_object(Bucket=_kb, Key=_kk)
        _kbytes = int(_kinfo["ContentLength"])
        if not (_kit_dest.is_file() and _kit_dest.stat().st_size == _kbytes
                and file_sha256(_kit_dest) == _want):
            _kpart = _kit_dest.with_name(_kit_dest.name + ".part")
            _kpart.unlink(missing_ok=True)
            print(f"Downloading {S3_KIT_SOURCE} ({_kbytes / 2**20:.1f} MiB) ...", flush=True)
            s3.download_file(_kb, _kk, str(_kpart),
                             Config=TransferConfig(**S3_TRANSFER))
            if _kpart.stat().st_size != _kbytes:
                raise IOError("Downloaded kit byte count differs from S3 ContentLength")
            if file_sha256(_kpart) != _want:
                _kpart.unlink(missing_ok=True)
                raise IOError("Downloaded kit SHA-256 does not match S3 sidecar")
            _kpart.replace(_kit_dest)
            print("Downloaded and SHA-256 verified:", _kit_dest, flush=True)
        else:
            print("Reusing SHA-256 verified kit bundle:", _kit_dest, flush=True)
        KIT_BUNDLE = _kit_dest
    except Exception as exc:
        raise FileNotFoundError(
            "ecg_image_v1_code.tar.gz not found locally and the S3 fallback failed "
            f"({type(exc).__name__}: {exc}). Searched: " +
            ", ".join(str(p) for p in _kit_search) +
            f"; S3: {S3_KIT_SOURCE}") from exc
_kit_sidecar = Path(str(KIT_BUNDLE) + ".sha256")
if not _kit_sidecar.is_file():
    raise FileNotFoundError(f"Kit SHA-256 sidecar missing: {_kit_sidecar}")
_kit_want = _kit_sidecar.read_text().split()[0].lower()
if len(_kit_want) != 64 or any(c not in "0123456789abcdef" for c in _kit_want):
    raise ValueError("Kit SHA-256 sidecar does not contain a valid digest")
KIT_BUNDLE_SHA256 = file_sha256(KIT_BUNDLE)
if KIT_BUNDLE_SHA256 != _kit_want:
    raise IOError("Kit bundle SHA-256 does not match its sidecar — re-transfer the file")
print("Kit bundle verified:", KIT_BUNDLE)

KIT_ROOT = WORK_DIR / "kit"
_kit_marker = KIT_ROOT / ".kit_extraction_complete.json"
_kit_marker_value = {"bundle_sha256": KIT_BUNDLE_SHA256,
                     "bundle_bytes": KIT_BUNDLE.stat().st_size}
_kit_ready = False
if _kit_marker.is_file():
    try:
        _kit_ready = json.loads(_kit_marker.read_text()) == _kit_marker_value and KIT_ROOT.is_dir()
    except (OSError, ValueError):
        pass
if _kit_ready:
    print("Reusing extracted kit:", KIT_ROOT)
else:
    _kit_marker.unlink(missing_ok=True)
    print("Extracting kit to", KIT_ROOT, "...")
    shutil.rmtree(KIT_ROOT, ignore_errors=True)
    with tarfile.open(KIT_BUNDLE, "r:gz") as _archive:
        try:
            _archive.extractall(KIT_ROOT, filter="data")
        except TypeError:  # Python without PEP 706 filters; bundle is SHA-256 verified
            _archive.extractall(KIT_ROOT)
    _kit_marker.write_text(json.dumps(_kit_marker_value, indent=2) + "\n")
    print("Kit extracted")

KIT_DIR = str(KIT_ROOT / "ecg_image_v1_code" / "ecg-image-generator")
_KIT_SCRIPT = Path(KIT_DIR) / "gen_ecg_images_from_data_batch.py"
if not _KIT_SCRIPT.is_file():
    raise FileNotFoundError(f"Kit batch driver missing: {_KIT_SCRIPT}")
if not (Path(KIT_DIR) / "Fonts" / KIT_FIXED_FONT).is_file():
    raise FileNotFoundError(f"Fixed font {KIT_FIXED_FONT} missing from kit Fonts/")

# Patch gate (PLAN §6): refuse to render from unpatched kit code.
_kit_need = {
    "ecg_plot.py": ["PATCHED by image_pipeline/patch_kit.py",
                    "PATCHED-LEADNAME", "PATCHED-RHYTHMNAME",
                    "PATCHED-INKWIDTH", "PATCHED-LAYOUT", "PATCHED-PAPER"],
    "extract_leads.py": ["PATCHED-COLUMNS"],
    "gen_ecg_images_from_data_batch.py": ["PATCHED-NUMPY2"],
    "gen_ecg_image_from_data.py": ["PATCHED-LAZYDEPS", "PATCHED-FIXEDFONT"],
    "helper_functions.py": ["PATCHED-UNUSED-IMGAUG"],
}
_kit_missing = []
for _name, _markers in _kit_need.items():
    _text = (Path(KIT_DIR) / _name).read_text()
    for _mk in _markers:
        if _mk not in _text:
            _kit_missing.append(f"{_name}:{_mk}")
if _kit_missing:
    raise RuntimeError(f"Kit at {KIT_DIR} is not patched "
                       f"({len(_kit_missing)} markers missing, e.g. {_kit_missing[:3]}); refusing to render")
print(f"Kit patch gate OK ({KIT_DIR})")

TEXTURE_DIR = Path(KIT_DIR) / "CreasesWrinkles" / "wrinkles-dataset"
_textures = sorted(p.name for p in TEXTURE_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
if len(_textures) != 19:
    raise RuntimeError(f"Expected the kit's 19 wrinkled-paper photographs, found {len(_textures)}")
print(f"{len(_textures)} wrinkled-paper photographs in {TEXTURE_DIR}")
STAGES_DONE.add("kit")

## 5. Discover and validate the 113,358 records

Records come from `manifest.csv`, never from globbing. Each header must be 12 leads × 5,000 samples at 500 Hz, and the label totals must equal `_checks/final_counts.csv`. These are the same checks the clean12 render notebook makes.

In [ ]:
with (INPUT_DIR / "manifest.csv").open(newline="", encoding="utf-8") as handle:
    source_rows = list(csv.DictReader(handle))
if len(source_rows) != EXPECTED_RECORDS or len({r["record_path"] for r in source_rows}) != EXPECTED_RECORDS:
    raise ValueError(f"manifest.csv lists {len(source_rows):,} rows / "
                     f"{len({r['record_path'] for r in source_rows}):,} unique; expected {EXPECTED_RECORDS:,}")

site_counts, label_totals, patients, records = Counter(), Counter(), set(), []
for row in source_rows:
    rel = PurePosixPath(row["record_path"])
    if (rel.is_absolute() or ".." in rel.parts or rel.suffix != ".hea"
            or rel.parts[0] != row["site"] or rel.parts[1] != "WFDB"):
        raise ValueError(f"Unexpected record_path: {row['record_path']!r}")
    header = INPUT_DIR.joinpath(*rel.parts)
    if not header.is_file():
        raise FileNotFoundError(f"Manifest record missing from extracted tree: {rel}")
    first = header.read_text(encoding="utf-8", errors="replace").split("\n", 1)[0].split()
    if (len(first) < 4 or first[1] != "12"
            or float(first[2].split("/")[0]) != EXPECTED_FS or int(first[3]) != EXPECTED_SAMPLES):
        raise ValueError(f"{rel}: header {' '.join(first[:4])!r} is not 12 x {EXPECTED_SAMPLES} @ {EXPECTED_FS} Hz")
    site_counts[row["site"]] += 1
    patients.add(row["patient_key"])
    for condition in CONDITIONS:
        label_totals[condition] += int(row[condition])
    records.append({"record_path": row["record_path"], "site": row["site"], "labels": row["labels"],
                    "image_path": (PurePosixPath("images") / rel.with_suffix(".png")).as_posix()})

with (INPUT_DIR / "_checks" / "final_counts.csv").open(newline="", encoding="utf-8") as handle:
    verified_totals = {row["condition"]: int(row["total"]) for row in csv.DictReader(handle)}
if verified_totals != dict(label_totals):
    raise ValueError(f"Label totals {dict(label_totals)} differ from _checks/final_counts.csv {verified_totals}")
record_by_path = {row["record_path"]: row for row in source_rows}
print(f"{len(records):,} records from {len(patients):,} patients | per site: {dict(sorted(site_counts.items()))}")
print("Records per condition:", dict(label_totals))
STAGES_DONE.add("records")

## 6. Worker modules

Two modules are written into `WORK_DIR`. The spawned pool processes import them, so `WORK_DIR` is put on `PYTHONPATH`.
- `ecg_clean_worker.py` is the clean12 render worker, verbatim: header parsing, millivolt loading, canonical `.dat` staging, the kit call, and the 1024x768 fit.
- `augment_worker.py` contains the stage-4 copy, the balanced planner, and the pool task. One task makes one kit call for a batch of records, then augments each grey sheet and writes it out.

In [ ]:
CLEAN_WORKER_SOURCE = r'''
"""Clean ECG worker: render with the real ECG-image-kit (PLAN §6).

Each batch is staged to canonical scratch WFDB .dat files (fmt 16, gain
1000, canonical lead order, int16 round-trip verified — this dodges the
kit's .mat gain bug, PLAN §0.5) and rendered by the kit batch driver
with the reference flags/env/fixed font. Grey masters are recoloured to
the canonical clean red grid (fixed LUT, PLAN §10) and resized LANCZOS
to 994x768 + 15 px paper-colour pads -> final 1024x768 PNG (PLAN §7.3).

Self-contained: stdlib + numpy + scipy + Pillow only. The kit itself
runs in a subprocess (kit_python, which carries the kit's own deps).
"""
import math
import os
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
from scipy.io import loadmat
from PIL import Image

LEADS = ("I", "II", "III", "aVR", "aVL", "aVF",
         "V1", "V2", "V3", "V4", "V5", "V6")
KIT_SCRIPT = "gen_ecg_images_from_data_batch.py"
KIT_TIMEOUT_S = 3600

# Clean-view recolour: fixed "red" palette (PLAN §10.3; port of
# v1_support.variants._lut with ink=(20,20,20), paper=red paper, sat=1.0).
GRID_MAJOR = (178.0, 86.0, 84.0)
GRID_MINOR = (228.0, 180.0, 176.0)
GRID_PAPER = (253.0, 249.0, 247.0)
GRID_INK = (20.0, 20.0, 20.0)
GRID_TRACE_MAX = 55
GRID_MAJOR_LEVEL = 102
GRID_MINOR_LEVEL = 191
GRID_PAPER_LEVEL = 250

FINAL_W, FINAL_H = 1024, 768
RESIZE_H = 768
MASTER_W, MASTER_H = 2200, 1700

CFG = None
INPUT = None
OUTPUT = None
_LUT = None


class InvalidRecordError(ValueError):
    pass


def init_worker(config, input_dir, output_dir):
    global CFG, INPUT, OUTPUT
    CFG = config
    INPUT = Path(input_dir)
    OUTPUT = Path(output_dir)
    for key in ("kit_dir", "kit_python", "kit_seed", "kit_fixed_font",
                "kit_render_flags", "kit_env", "kit_scratch_dir",
                "max_duration_s", "png_compress_level"):
        if key not in CFG:
            raise ValueError(f"render_config missing {key!r}")
    if not (Path(CFG["kit_dir"]) / KIT_SCRIPT).is_file():
        raise ValueError("kit batch driver missing — rerun the kit-bundle cell")


def _lut():
    major = np.asarray(GRID_MAJOR, dtype=np.float32)
    minor = np.asarray(GRID_MINOR, dtype=np.float32)
    paper = np.asarray(GRID_PAPER, dtype=np.float32)
    ink = np.asarray(GRID_INK, dtype=np.float32)

    def _sat(c):
        lum = float(np.dot(c, (0.299, 0.587, 0.114)))
        return np.clip(lum + (c - lum) * 1.0, 0, 255)

    major, minor = _sat(major), _sat(minor)
    xs = np.array([0, GRID_TRACE_MAX, GRID_MAJOR_LEVEL,
                   GRID_MINOR_LEVEL, GRID_PAPER_LEVEL, 255], dtype=np.float32)
    lut = np.empty((256, 3), dtype=np.float32)
    grid = np.arange(256, dtype=np.float32)
    for ch in range(3):
        ys = np.array([ink[ch] * 0.35, ink[ch], major[ch], minor[ch],
                       paper[ch], min(255.0, paper[ch] + 3)], dtype=np.float32)
        lut[:, ch] = np.interp(grid, xs, ys)
    return np.clip(lut, 0, 255)


def recolour_clean(master):
    """Fixed clean-red recolour of a grey kit master.

    The float LUT is truncated to uint8 once (what the kit-v1 per-pixel
    `_LUT[grey].astype(np.uint8)` did) and applied through Pillow point
    tables: bit-identical output, ~7x cheaper per image.
    """
    global _LUT
    if _LUT is None:
        lut = _lut().astype(np.uint8)
        _LUT = [lut[:, ch].tolist() for ch in range(3)]
    grey = master.convert("RGB").convert("L")
    return Image.merge("RGB", [grey.point(table) for table in _LUT])


def _paper_colour(arr):
    # 90th percentile of corner patches (not median): antialiased grid-halo
    # pixels drag a median toward tint; p90 recovers the paper white.
    h, w = arr.shape[:2]
    k = max(8, min(h, w) // 40)
    patches = np.concatenate([
        arr[:k, :k].reshape(-1, 3), arr[:k, -k:].reshape(-1, 3),
        arr[-k:, :k].reshape(-1, 3), arr[-k:, -k:].reshape(-1, 3)])
    return tuple(int(v) for v in np.percentile(patches, 90, axis=0))


def to_final(img):
    """LANCZOS fit to height 768 + symmetric paper-colour pad to 1024x768."""
    w, h = img.size
    scale = RESIZE_H / h
    new_w = round(w * scale)
    small = img.resize((new_w, RESIZE_H), Image.LANCZOS)
    if abs((w / h) - (new_w / RESIZE_H)) > 0.5 / RESIZE_H + 1e-9:
        raise InvalidRecordError(
            f"aspect drift: source {w}x{h} -> {new_w}x{RESIZE_H}")
    pad_total = FINAL_W - new_w
    if pad_total < 0:
        raise InvalidRecordError(
            f"resized width {new_w} exceeds final {FINAL_W}")
    left = pad_total // 2
    fill = _paper_colour(np.asarray(small.convert("RGB"), dtype=np.float32))
    canvas = Image.new("RGB", (FINAL_W, FINAL_H), fill)
    canvas.paste(small.convert("RGB"), (left, 0))
    if canvas.size != (FINAL_W, FINAL_H):
        raise InvalidRecordError(f"final size {canvas.size}")
    return canvas


def parse_header(path):
    try:
        lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    except OSError as exc:
        raise InvalidRecordError(f"Unreadable header: {exc}") from exc
    if not lines:
        raise InvalidRecordError("Empty header")
    first = lines[0].split()
    if len(first) < 4 or "/" in first[0] or first[0] != path.stem:
        raise InvalidRecordError("Malformed/multi-segment record line")
    try:
        nleads = int(first[1])
        fs = float(first[2].split("/")[0])
        nsamp = int(first[3])
    except ValueError as exc:
        raise InvalidRecordError("Invalid record dimensions") from exc
    if nleads != 12 or not math.isfinite(fs) or fs <= 0 or nsamp <= 0:
        raise InvalidRecordError(
            f"Unsupported dimensions: {nleads} leads, fs={fs}, n={nsamp}")
    duration = nsamp / fs
    if duration > CFG["max_duration_s"] + 1e-6:
        raise InvalidRecordError(
            f"Duration {duration:.4f}s exceeds configured maximum")
    rows = []
    comments = {}
    for line in lines[1:]:
        if line.startswith("#"):
            body = line[1:].strip()
            if ":" in body:
                k, v = body.split(":", 1)
                comments[k.strip()] = v.strip()
            continue
        if not line.strip():
            continue
        f = line.split()
        if len(f) < 9:
            raise InvalidRecordError("Malformed signal line")
        name = f[0]
        if name != Path(name).name or name in (".", "..", "~"):
            raise InvalidRecordError("Unsafe signal filename")
        fmt = f[1]
        if fmt not in ("16", "16+24", "16x1+24"):
            raise InvalidRecordError(f"Unsupported signal format {fmt}")
        gain_field = f[2]
        if not gain_field.endswith("/mV"):
            raise InvalidRecordError(f"Expected /mV units, got {gain_field}")
        try:
            gain = float(gain_field.split("(")[0].split("/")[0])
            adc_zero = int(f[4])
        except ValueError as exc:
            raise InvalidRecordError("Invalid gain or ADC zero") from exc
        if not math.isfinite(gain) or gain <= 0:
            raise InvalidRecordError("Nonpositive/nonfinite ADC gain")
        if "(" in gain_field and gain_field.split("(", 1)[1].split(")", 1)[0] != str(adc_zero):
            raise InvalidRecordError("Header baseline/ADC zero disagree")
        rows.append({"file": name, "fmt": fmt, "gain": gain,
                     "zero": adc_zero, "lead": f[-1]})
    if len(rows) != 12:
        raise InvalidRecordError("Need exactly 12 signal lines")
    upper = {lead.upper(): lead for lead in LEADS}
    names = []
    for r in rows:
        key = r["lead"].strip().upper()
        if key not in upper:
            raise InvalidRecordError(f"Non-canonical lead {r['lead']!r}")
        names.append(upper[key])
    if sorted(names) != sorted(LEADS):
        raise InvalidRecordError("Lead set is not the canonical 12")
    if len({r["file"] for r in rows}) != 1 or len({r["fmt"] for r in rows}) != 1:
        raise InvalidRecordError("Multiple signal files/formats not supported")
    return rows, names, fs, nsamp, duration, comments


def load_mv(path, rows, names, nsamp):
    fmt = rows[0]["fmt"]
    sig_path = path.parent / rows[0]["file"]
    if not sig_path.is_file():
        raise InvalidRecordError(f"Missing signal file {rows[0]['file']}")
    if fmt == "16":
        n = 12 * nsamp
        raw = np.fromfile(str(sig_path), dtype="<i2", count=n).astype(np.int64)
        if raw.size != n:
            raise InvalidRecordError(
                f".dat holds {raw.size} samples, need {n}")
        raw = raw.reshape(nsamp, 12).T
    else:
        try:
            arr = np.asarray(loadmat(str(sig_path), variable_names=["val"])["val"])
        except (OSError, ValueError, KeyError, TypeError) as exc:
            raise InvalidRecordError(f"Unreadable MAT val: {exc}") from exc
        if arr.shape == (nsamp, 12):
            arr = arr.T
        if arr.shape != (12, nsamp) or arr.dtype.kind not in "iu":
            raise InvalidRecordError(
                f"Invalid MAT val shape/type {arr.shape}/{arr.dtype}")
        raw = arr.astype(np.int64)
    if raw.shape != (12, nsamp):
        raise InvalidRecordError(f"Unexpected signal shape {raw.shape}")
    pos = {name: i for i, name in enumerate(names)}
    order = [pos[lead] for lead in LEADS]
    gains = np.array([rows[pos[lead]]["gain"] for lead in LEADS],
                     dtype=np.float64)[:, None]
    zeros = np.array([rows[pos[lead]]["zero"] for lead in LEADS],
                     dtype=np.float64)[:, None]
    # Necessary unit conversion only: digital ADC counts -> physical millivolts.
    mv = (raw[order, :].astype(np.float64) - zeros) / gains
    if not np.isfinite(mv).all():
        raise InvalidRecordError("Nonfinite physical signal")
    return mv


def stage_record(record_id, mv, fs, comments, staged_dir):
    """Write canonical scratch .dat + .hea for the kit; verify round-trip."""
    if mv.shape[0] != 12:
        raise InvalidRecordError(f"shape {mv.shape}, need (12, N)")
    n = mv.shape[1]
    raw = np.round(mv * 1000.0)
    if raw.min() < -32768 or raw.max() > 32767:
        raise InvalidRecordError("signal exceeds int16 at gain 1000")
    staged_dir.mkdir(parents=True, exist_ok=True)
    hea = staged_dir / f"{record_id}.hea"
    dat = staged_dir / f"{record_id}.dat"
    np.ascontiguousarray(raw.astype("<i2").T).tofile(str(dat))
    rec_fs = int(fs) if float(fs).is_integer() else fs
    lines = [f"{record_id} 12 {rec_fs} {n}"]
    for lead in LEADS:
        lines.append(f"{record_id}.dat 16 1000.0(0)/mV 16 0 0 0 0 {lead}")
    for key in ("Age", "Sex"):
        if comments.get(key):
            lines.append(f"# {key}: {comments[key]}")
    hea.write_text("\n".join(lines) + "\n")
    back = np.fromfile(str(dat), dtype="<i2",
                       count=12 * n).astype(np.int64).reshape(n, 12).T
    if back.shape != raw.shape or not np.array_equal(back, raw.astype(np.int64)):
        raise InvalidRecordError("staging round-trip mismatch")
    return hea


def run_kit(staged_dir, render_dir):
    """Invoke the kit batch driver once for a staged directory."""
    kit_dir = Path(CFG["kit_dir"])
    if not (kit_dir / KIT_SCRIPT).is_file():
        raise RuntimeError(f"kit script missing: {kit_dir / KIT_SCRIPT}")
    render_dir.mkdir(parents=True, exist_ok=True)
    # NOTE: absolute paths are mandatory — the kit subprocess runs with
    # cwd=kit_dir, so relative -i/-o would resolve inside the kit tree.
    cmd = [CFG["kit_python"], KIT_SCRIPT,
           "-i", str(Path(staged_dir).resolve()),
           "-o", str(Path(render_dir).resolve()),
           "-se", str(CFG["kit_seed"]), *CFG["kit_render_flags"]]
    env = dict(os.environ)
    env.update(CFG.get("kit_env", {}))
    env["ECG_FIXED_FONT"] = CFG["kit_fixed_font"]
    # one numerical thread per renderer; never the GPU
    env.update({"CUDA_VISIBLE_DEVICES": "", "OMP_NUM_THREADS": "1",
                "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"})
    proc = subprocess.run(cmd, cwd=str(kit_dir), capture_output=True,
                          text=True, timeout=KIT_TIMEOUT_S, env=env)
    if proc.returncode != 0:
        tail = (proc.stderr or proc.stdout or "").strip().splitlines()[-3:]
        raise RuntimeError(f"kit batch failed "
                           f"({staged_dir} -> {render_dir}): {'; '.join(tail)}")


def collect_pngs(render_dir, record_ids):
    """Map record_id -> rendered PNG. The kit names outputs <record>-0.png."""
    found = {}
    for png in Path(render_dir).rglob("*.png"):
        stem = png.stem  # <record_id>-0
        rid = stem[:-2] if stem.endswith("-0") else stem
        if rid in record_ids:
            found[rid] = png
    missing = sorted(set(record_ids) - set(found))
    if missing:
        raise RuntimeError(f"kit produced no PNG for {len(missing)} records, "
                           f"e.g. {missing[:5]}")
    return found


def clean_render_dir(render_dir):
    """Remove kit side products; only *.png may remain."""
    for pat in ("*.hea", "*.dat", "*.json", "*.csv"):
        for p in Path(render_dir).rglob(pat):
            try:
                p.unlink()
            except OSError:
                pass


def record_id_for(rel):
    stem = Path(rel).with_suffix("").as_posix()
    rid = stem.replace("/", "__").replace("\\", "__")
    if not rid or rid in (".", "..") or "/" in rid or len(rid) > 180:
        raise InvalidRecordError(f"Unusable record id for {rel!r}")
    return rid


def _base_row(rel):
    return {"source_header": rel,
            "image_path": str(Path("images") / Path(rel).with_suffix(".png")),
            "status": "", "error": "", "sample_rate_hz": "", "samples": "",
            "duration_s": "", "image_width": "", "image_height": "",
            "image_bytes": ""}


def process_batch(batch):
    scratch = (Path(CFG["kit_scratch_dir"])
               / f"w{os.getpid()}-{int(time.time() * 1000) % 100000000}")
    staged = scratch / "staged"
    render = scratch / "render"
    rows = {}
    todo = {}
    for rel in batch:
        base = _base_row(rel)
        rows[rel] = base
        header = INPUT / rel
        try:
            prows, names, fs, nsamp, duration, comments = parse_header(header)
            base.update(sample_rate_hz=fs, samples=nsamp,
                        duration_s=round(duration, 6))
            out = OUTPUT / base["image_path"]
            if out.is_file() and out.stat().st_size > 100:
                try:
                    with Image.open(out) as existing:
                        ok = existing.size == (FINAL_W, FINAL_H)
                        existing.verify()
                    if ok:
                        base.update(status="skipped", image_width=FINAL_W,
                                    image_height=FINAL_H,
                                    image_bytes=out.stat().st_size)
                        continue
                except (OSError, ValueError):
                    pass
                # A mismatched or damaged existing image is regenerated.
            mv = load_mv(header, prows, names, nsamp)
            rid = record_id_for(rel)
            if rid in todo:
                raise InvalidRecordError(f"Duplicate staged id {rid}")
            todo[rid] = (rel, mv, fs, comments)
        except InvalidRecordError as exc:
            base.update(status="invalid", error=str(exc)[:1000])
        except Exception as exc:
            base.update(status="failed",
                        error=f"{type(exc).__name__}: {exc}"[:1000])
    try:
        if todo:
            for rid, (rel, mv, fs, comments) in todo.items():
                if rows[rel]["status"]:
                    continue
                try:
                    stage_record(rid, mv, fs, comments, staged)
                except InvalidRecordError as exc:
                    rows[rel].update(status="invalid", error=str(exc)[:1000])
            staged_ids = [rid for rid in todo if not rows[todo[rid][0]]["status"]]
            if staged_ids:
                try:
                    run_kit(staged, render)
                    got = collect_pngs(render, set(staged_ids))
                except Exception as exc:
                    for rid in staged_ids:
                        rows[todo[rid][0]].update(
                            status="failed",
                            error=f"{type(exc).__name__}: {exc}"[:1000])
                    got = {}
                for rid, png in got.items():
                    rel = todo[rid][0]
                    base = rows[rel]
                    try:
                        with Image.open(png) as master:
                            if master.size != (MASTER_W, MASTER_H):
                                raise InvalidRecordError(
                                    f"master size {master.size}, "
                                    f"need {(MASTER_W, MASTER_H)}")
                            view = recolour_clean(master)
                        final = to_final(view)
                        out = OUTPUT / base["image_path"]
                        out.parent.mkdir(parents=True, exist_ok=True)
                        temp = out.with_name(out.name + f".{os.getpid()}.part")
                        try:
                            final.save(temp, format="PNG",
                                       compress_level=CFG["png_compress_level"])
                            temp.replace(out)
                        finally:
                            try:
                                temp.unlink(missing_ok=True)
                            except OSError:
                                pass
                            final.close()
                        base.update(status="success", image_width=FINAL_W,
                                    image_height=FINAL_H,
                                    image_bytes=out.stat().st_size)
                    except InvalidRecordError as exc:
                        base.update(status="invalid", error=str(exc)[:1000])
                    except Exception as exc:
                        base.update(status="failed",
                                    error=f"{type(exc).__name__}: {exc}"[:1000])
                clean_render_dir(render)
    finally:
        shutil.rmtree(scratch, ignore_errors=True)
    return [rows[rel] for rel in batch]


def process_one(rel):
    return process_batch([rel])[0]


def ping():
    """Zero-cost import probe: lets §7 verify a spawn-context pool child
    can import this module without rendering anything."""
    return {"ok": True, "leads": list(LEADS),
            "final": (FINAL_W, FINAL_H), "master": (MASTER_W, MASTER_H)}

'''

AUGMENT_WORKER_SOURCE = r'''"""Augmented clean12 worker: one paper/photo sheet per clean12 ECG record.

Each record is rendered by the clean12 render worker (ecg_clean_worker, the
module embedded in Clean12_ECG_Signal_Images_EC2.ipynb: canonical WFDB
staging + the patched ECG-image-kit with the clean12 flags) to its 2200x1700
grey kit master, exactly as for the clean12 sheets. The master then goes
through the image pipeline's stage-4 paper/photo augmentation
(dataset-combination-code/ecg-dataset-pipeline/image_pipeline/stage4_augment.py,
the code behind the correct_prediction_samples previews), a JPEG q85 round
trip, and the clean view's own 1024x768 fit (ecg_clean_worker.to_final).

The stage-4 functions below are a line-for-line port (checked pixel for
pixel against stage4_augment.augment with the stage-4 settings). Two changes:
wrinkled paper and fold creases are stronger than stage 4 (WRINKLE/CREASE
settings below, raised on request after comparing previews), and with a plan
the balanced factors (grid palette, ink strength, tilt, perspective, fold
creases, edge shadow, blur) are forced to the planned level. Every other
value keeps its stage-4 distribution.
"""
from __future__ import annotations

import hashlib
import io
import os
import shutil
import time
from pathlib import Path

import numpy as np
from PIL import Image, ImageFilter

import ecg_clean_worker as R

# ---------------------------------------------------------------- stage-4 config
# Values copied from image_pipeline/config.py (the corpus stage-4 settings),
# except the wrinkle/crease block marked STRONGER.
STAGE4_SEED = 20260818
GRID_PALETTES = {
    # major gridline        minor gridline        paper
    "red":    ((178, 86, 84), (228, 180, 176), (253, 249, 247)),
    "pink":   ((188, 124, 148), (236, 200, 212), (254, 251, 252)),
    "orange": ((186, 126, 86), (236, 202, 176), (254, 250, 245)),
    "grey":   ((124, 124, 124), (198, 198, 198), (252, 252, 252)),
}
GRID_PALETTE_WEIGHTS = {"red": 0.38, "orange": 0.22, "pink": 0.20, "grey": 0.20}
GRID_SATURATION = (0.72, 1.02)
GRID_TRACE_MAX = 55
GRID_MAJOR_LEVEL = 102
GRID_MINOR_LEVEL = 191
GRID_PAPER_LEVEL = 250
TILT_RANGE = (-3.0, 3.0)
KEYSTONE_PROB = 0.7
KEYSTONE_MAG = (0.004, 0.018)
# STRONGER than stage 4 (stage 4: texture 0.30-0.62 clip 0.86-1.10; crease
# amp 0.06-0.13 clip 0.88-1.14 width 0.006-0.020; bow 0.02-0.06). Wrinkles
# read clearly and folds show as sharp ridges; the darkest fold still sits far
# above the trace ink, so traces stay legible.
TEXTURE_STRENGTH = (0.65, 1.00)
TEXTURE_CLIP = (0.76, 1.14)
CREASE_PROB = 0.62               # stage-4 draw; the balanced plan sets creases on/off
CREASE_AMP = (0.16, 0.26)
CREASE_CLIP = (0.76, 1.22)
CREASE_WIDTH = (0.004, 0.012)   # fraction of the long side
BOW_AMP = (0.03, 0.08)
EDGE_SHADOW_PROB = 0.65
PAPER_LEVEL_TARGET = (216, 248)
DEFOCUS_PROB = 0.55
DEFOCUS_RADIUS = (0.3, 0.9)
MARGIN_PX = 18
JPEG_QUALITY = 85
JPEG_SUBSAMPLING = 2  # 4:2:0

# ---------------------------------------------------------------- balanced plan
AUG_PALETTES = ("orange", "pink", "grey")  # red is the clean view's grid
INK_BINS = ((0.72, 0.82), (0.82, 0.92), (0.92, 1.02))
TILT_BINS = ((-3.0, -2.0), (-2.0, -1.0), (-1.0, 0.0), (0.0, 1.0), (1.0, 2.0), (2.0, 3.0))
PLAN_FACTORS = {
    "grid_palette": AUG_PALETTES,
    "ink_bin": tuple(range(len(INK_BINS))),
    "tilt_bin": tuple(range(len(TILT_BINS))),
    "perspective": (0, 1),
    "fold_creases": (0, 1),
    "edge_shadow": (0, 1),
    "blur": (0, 1),
}

MANIFEST_FIELDS = [
    "record_path", "image_path", "status", "error", "image_bytes",
    "grid_palette", "ink_bin", "grid_saturation", "ink_level", "tilt_bin", "tilt_deg",
    "perspective", "perspective_mag", "wrinkle_texture", "wrinkle_strength",
    "fold_creases", "fold_lines", "paper_bows", "light_angle_deg", "light_ramp",
    "corner_falloff", "edge_shadow", "shadow_edge", "shadow_depth", "shadow_width",
    "tint_rgb_gain", "brightness", "contrast", "paper_level", "sensor_noise_sigma",
    "blur", "blur_radius_px",
]

TEXTURE_DIR: Path | None = None
_TEXTURES: tuple[Path, ...] | None = None


def set_texture_dir(path) -> None:
    global TEXTURE_DIR, _TEXTURES
    TEXTURE_DIR, _TEXTURES = Path(path), None


def _texture_files() -> tuple[Path, ...]:
    """The kit's wrinkled-paper photographs (CreasesWrinkles/wrinkles-dataset)."""
    global _TEXTURES
    if _TEXTURES is None:
        if TEXTURE_DIR is None or not TEXTURE_DIR.is_dir():
            raise RuntimeError(f"wrinkle texture folder missing: {TEXTURE_DIR}")
        _TEXTURES = tuple(sorted(p for p in TEXTURE_DIR.iterdir()
                                 if p.suffix.lower() in {".jpg", ".jpeg", ".png"}))
        if not _TEXTURES:
            raise RuntimeError(f"no wrinkle textures in {TEXTURE_DIR}")
    return _TEXTURES


def blake_int(*parts) -> int:
    key = "|".join(str(p) for p in parts).encode()
    return int.from_bytes(hashlib.blake2b(key, digest_size=8).digest(), "big")


def stage4_rng(seed_parts: tuple) -> np.random.Generator:
    """stage4_augment._rng: per-image generator keyed by output identity only."""
    return np.random.default_rng(blake_int(*seed_parts) ^ STAGE4_SEED)


# ---------------------------------------------------------------- stage-4 port
def _paper_colour(img: np.ndarray) -> np.ndarray:
    h, w = img.shape[:2]
    k = max(8, min(h, w) // 40)
    patches = np.concatenate([
        img[:k, :k].reshape(-1, 3), img[:k, -k:].reshape(-1, 3),
        img[-k:, :k].reshape(-1, 3), img[-k:, -k:].reshape(-1, 3)])
    return np.median(patches, axis=0)


def _rotate_on_paper(img: Image.Image, deg: float, fill) -> Image.Image:
    w, h = img.size
    big = img.rotate(deg, resample=Image.BICUBIC, expand=True, fillcolor=fill)
    bw, bh = big.size
    left, top = (bw - w) // 2, (bh - h) // 2
    return big.crop((left, top, left + w, top + h))


def _illumination(arr, rng, meta, shadow=None):
    h, w = arr.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    ang = rng.uniform(0, 2 * np.pi)
    ramp = (np.cos(ang) * (xx / w - 0.5) + np.sin(ang) * (yy / h - 0.5))
    ramp_amp = rng.uniform(0.10, 0.22)
    field = 1.0 + ramp_amp * ramp * 2.0
    r = np.sqrt((xx / w - 0.5) ** 2 + (yy / h - 0.5) ** 2)
    falloff = rng.uniform(0.06, 0.18)
    field *= 1.0 - falloff * (r / r.max()) ** 2
    drawn = rng.random() < EDGE_SHADOW_PROB
    on = drawn if shadow is None else bool(shadow)
    meta.update(light_angle_deg=round(float(np.degrees(ang)), 1),
                light_ramp=round(float(ramp_amp), 4), corner_falloff=round(float(falloff), 4),
                edge_shadow=int(on), shadow_edge="", shadow_depth="", shadow_width="")
    if on:
        edge = int(rng.integers(0, 4))
        d = {0: xx / w, 1: 1 - xx / w, 2: yy / h, 3: 1 - yy / h}[edge]
        depth, width = rng.uniform(0.10, 0.26), rng.uniform(0.04, 0.12)
        field *= 1.0 - depth * np.exp(-d / width)
        meta.update(shadow_edge=("left", "right", "top", "bottom")[edge],
                    shadow_depth=round(float(depth), 4), shadow_width=round(float(width), 4))
    return arr * field[..., None]


def _paper_texture(arr, rng, meta):
    files = _texture_files()
    h, w = arr.shape[:2]
    pick = files[int(rng.integers(len(files)))]
    tex = Image.open(pick).convert("L")
    tw, th = tex.size
    scale = rng.uniform(0.55, 1.0)
    cw, ch = max(16, int(tw * scale)), max(16, int(th * scale))
    x0 = int(rng.integers(0, max(1, tw - cw + 1)))
    y0 = int(rng.integers(0, max(1, th - ch + 1)))
    tex = tex.crop((x0, y0, x0 + cw, y0 + ch))
    if rng.random() < 0.5:
        tex = tex.transpose(Image.FLIP_LEFT_RIGHT)
    if rng.random() < 0.5:
        tex = tex.transpose(Image.FLIP_TOP_BOTTOM)
    if rng.random() < 0.5:
        tex = tex.transpose(Image.ROTATE_90)
    field = np.asarray(tex.resize((w, h), Image.LANCZOS), dtype=np.float32) / 255.0
    mean = float(field.mean())
    if mean <= 1e-3:
        raise RuntimeError(f"blank wrinkle texture {pick.name}")
    strength = rng.uniform(*TEXTURE_STRENGTH)
    meta.update(wrinkle_texture=pick.name, wrinkle_strength=round(float(strength), 4))
    field = 1.0 + (field / mean - 1.0) * strength
    return arr * np.clip(field, *TEXTURE_CLIP)[..., None]


def _creases(arr, rng, meta):
    h, w = arr.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    field = np.ones((h, w), dtype=np.float32)
    folds = []
    for _ in range(int(rng.integers(1, 4))):
        if rng.random() < 0.5:
            pos = rng.uniform(0.12, 0.88) * h
            tilt = rng.uniform(-0.06, 0.06)
            dist = yy - (pos + tilt * (xx - w / 2))
            folds.append("h")
        else:
            pos = rng.uniform(0.12, 0.88) * w
            tilt = rng.uniform(-0.06, 0.06)
            dist = xx - (pos + tilt * (yy - h / 2))
            folds.append("v")
        width = rng.uniform(*CREASE_WIDTH) * max(h, w)
        amp = rng.uniform(*CREASE_AMP)
        g = np.exp(-(dist ** 2) / (2 * width ** 2))
        field *= 1.0 + amp * g * (dist / width) / 1.6
    bows = int(rng.integers(1, 3))
    for _ in range(bows):
        k = rng.uniform(0.6, 1.8)
        phase = rng.uniform(0, 2 * np.pi)
        axis = yy / h if rng.random() < 0.5 else xx / w
        field *= 1.0 + rng.uniform(*BOW_AMP) * np.sin(2 * np.pi * k * axis + phase)
    meta.update(fold_lines="".join(folds), paper_bows=bows)
    return arr * np.clip(field, *CREASE_CLIP)[..., None]


def _normalise_paper(arr, rng, meta):
    lum = arr @ np.array([0.299, 0.587, 0.114], dtype=np.float32)
    paper = float(np.percentile(lum, 95))
    if paper <= 1.0:
        return arr
    target = rng.uniform(*PAPER_LEVEL_TARGET)
    meta["paper_level"] = round(float(target), 2)
    return arr * (target / paper)


def _colour_cast(arr, rng, meta):
    gains = 1.0 + rng.normal(0.0, 0.012, size=3)
    meta["tint_rgb_gain"] = "/".join(f"{g:.4f}" for g in gains)
    return arr * gains[None, None, :]


def _grid_lut(palette, rng, sat_bin=None):
    major, minor, paper = (np.asarray(c, dtype=np.float32) for c in palette)
    ink = np.full(3, rng.uniform(8, 34), dtype=np.float32)
    ink += rng.normal(0, 3, size=3)
    paper = paper * rng.uniform(0.985, 1.005) + rng.normal(0, 1.2, size=3)
    sat = rng.uniform(*GRID_SATURATION)
    if sat_bin is not None:  # same draw, mapped into the planned bin
        lo, hi = INK_BINS[sat_bin]
        u = (sat - GRID_SATURATION[0]) / (GRID_SATURATION[1] - GRID_SATURATION[0])
        sat = lo + u * (hi - lo)

    def _sat(c):
        lum = float(np.dot(c, (0.299, 0.587, 0.114)))
        return np.clip(lum + (c - lum) * sat, 0, 255)

    major, minor = _sat(major), _sat(minor)
    xs = np.array([0, GRID_TRACE_MAX, GRID_MAJOR_LEVEL, GRID_MINOR_LEVEL,
                   GRID_PAPER_LEVEL, 255], dtype=np.float32)
    lut = np.empty((256, 3), dtype=np.float32)
    grid = np.arange(256, dtype=np.float32)
    for ch in range(3):
        ys = np.array([ink[ch] * 0.35, ink[ch], major[ch], minor[ch], paper[ch],
                       min(255.0, paper[ch] + 3)], dtype=np.float32)
        lut[:, ch] = np.interp(grid, xs, ys)
    return np.clip(lut, 0, 255), float(sat), float(ink.mean())


def _recolour_grid(master: Image.Image, rng, force=None, sat_bin=None):
    names = list(GRID_PALETTES)
    weights = np.array([GRID_PALETTE_WEIGHTS.get(n, 0.0) for n in names], dtype=np.float64)
    weights = weights / weights.sum()
    drawn = names[int(rng.choice(len(names), p=weights))]  # draw, THEN override (stage 4)
    name = force if force in GRID_PALETTES else drawn
    lut, sat, ink = _grid_lut(GRID_PALETTES[name], rng, sat_bin)
    out = Image.fromarray(lut[np.asarray(master.convert("L"))].astype(np.uint8), "RGB")
    return out, name, sat, ink


def _defocus(img, rng, meta, blur=None):
    drawn = rng.random() < DEFOCUS_PROB
    on = drawn if blur is None else bool(blur)
    meta.update(blur=int(on), blur_radius_px="")
    if not on:
        return img
    radius = float(rng.uniform(*DEFOCUS_RADIUS))
    meta["blur_radius_px"] = round(radius, 4)
    return img.filter(ImageFilter.GaussianBlur(radius=radius))


def augment(master: Image.Image, rng: np.random.Generator, plan: dict | None = None):
    """stage4_augment.augment on a grey kit master (RGB), in memory.

    Returns (RGB sheet with the stage-4 margin, effect metadata). plan keys are
    PLAN_FACTORS; plan=None reproduces stage 4 draw for draw.
    """
    plan = plan or {}
    meta: dict = {}
    img, palette, sat, ink = _recolour_grid(master, rng, plan.get("grid_palette"), plan.get("ink_bin"))
    meta.update(grid_palette=palette, grid_saturation=round(sat, 4), ink_level=round(ink, 2))
    arr0 = np.asarray(img, dtype=np.float32)
    fill = tuple(int(v) for v in _paper_colour(arr0))

    deg = float(rng.uniform(*TILT_RANGE))
    if "tilt_bin" in plan:
        lo, hi = TILT_BINS[plan["tilt_bin"]]
        deg = lo + (deg - TILT_RANGE[0]) / (TILT_RANGE[1] - TILT_RANGE[0]) * (hi - lo)
    meta["tilt_deg"] = round(deg, 3)
    img = _rotate_on_paper(img, deg, fill)

    keystone = rng.random() < KEYSTONE_PROB
    if "perspective" in plan:
        keystone = bool(plan["perspective"])
    meta.update(perspective=int(keystone), perspective_mag="")
    if keystone:
        w, h = img.size
        m = rng.uniform(*KEYSTONE_MAG)
        dx, dy = m * w, m * h
        quad = (rng.uniform(0, dx), rng.uniform(0, dy),
                rng.uniform(0, dx), h - rng.uniform(0, dy),
                w - rng.uniform(0, dx), h - rng.uniform(0, dy),
                w - rng.uniform(0, dx), rng.uniform(0, dy))
        meta["perspective_mag"] = round(float(m), 5)
        img = img.transform((w, h), Image.QUAD, quad, resample=Image.BICUBIC, fillcolor=fill)

    arr = np.asarray(img, dtype=np.float32)
    arr = _paper_texture(arr, rng, meta)
    creases = rng.random() < CREASE_PROB
    if "fold_creases" in plan:
        creases = bool(plan["fold_creases"])
    meta.update(fold_creases=int(creases), fold_lines="", paper_bows="")
    if creases:
        arr = _creases(arr, rng, meta)
    arr = _illumination(arr, rng, meta, plan.get("edge_shadow"))
    arr = _colour_cast(arr, rng, meta)
    exposure = rng.uniform(0.92, 1.06)
    arr *= exposure
    mean = arr.mean()
    contrast = rng.uniform(0.94, 1.08)
    arr = (arr - mean) * contrast + mean
    meta.update(brightness=round(float(exposure), 4), contrast=round(float(contrast), 4))
    arr = _normalise_paper(arr, rng, meta)
    noise = rng.uniform(1.5, 4.5)
    meta["sensor_noise_sigma"] = round(float(noise), 3)
    arr += rng.normal(0.0, noise, size=arr.shape)
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    out = _defocus(Image.fromarray(arr), rng, meta, plan.get("blur"))

    m = MARGIN_PX
    canvas = Image.new("RGB", (out.width + 2 * m, out.height + 2 * m), fill)
    canvas.paste(out, (m, m))
    return canvas, meta


def jpeg_roundtrip(img: Image.Image) -> Image.Image:
    """Stage 4 writes its photograph as JPEG q85 4:2:0; keep those artifacts."""
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=JPEG_QUALITY, subsampling=JPEG_SUBSAMPLING)
    buf.seek(0)
    out = Image.open(buf)
    out.load()
    return out


def augment_master(master: Image.Image, seed: int, record_path: str, plan: dict):
    """Grey kit master -> final 1024x768 augmented sheet + metadata."""
    sheet, meta = augment(master.convert("RGB"), stage4_rng((seed, record_path, "clean12-aug")), plan)
    return R.to_final(jpeg_roundtrip(sheet)), meta


def check_plan(meta: dict, plan: dict) -> None:
    """Raise unless the rendered effects are exactly the planned levels."""
    for factor in ("grid_palette", "perspective", "fold_creases", "edge_shadow", "blur"):
        if meta[factor] != plan[factor]:
            raise RuntimeError(f"{factor}: planned {plan[factor]!r}, rendered {meta[factor]!r}")
    lo, hi = TILT_BINS[plan["tilt_bin"]]
    if not lo - 1e-3 <= meta["tilt_deg"] <= hi + 1e-3:
        raise RuntimeError(f"tilt {meta['tilt_deg']} outside planned bin {lo}..{hi}")
    lo, hi = INK_BINS[plan["ink_bin"]]
    if not lo - 1e-3 <= meta["grid_saturation"] <= hi + 1e-3:
        raise RuntimeError(f"ink strength {meta['grid_saturation']} outside planned bin {lo}..{hi}")


# ---------------------------------------------------------------- balanced plan
def balanced_plan(records: list[dict], seed: int) -> dict[str, dict]:
    """Assign every factor level equally, independently of the labels.

    records: dicts with record_path, site, labels. For each factor the records
    are ordered by stratum (site, label set) and, inside a stratum, by a
    per-factor seeded hash; levels are dealt round-robin along that order with
    one counter that runs across strata. So every factor is exactly balanced
    overall (counts differ by at most 1), balanced to +-1 inside every label
    combination, and paired with the other factors by independent shuffles.
    """
    keyed = sorted(records, key=lambda r: (r["site"], r["labels"], r["record_path"]))
    plan = {r["record_path"]: {} for r in keyed}
    for factor, levels in PLAN_FACTORS.items():
        order = sorted(keyed, key=lambda r: (r["site"], r["labels"],
                                            blake_int(seed, factor, r["record_path"])))
        offset = blake_int(seed, factor, "offset") % len(levels)
        for i, r in enumerate(order):
            plan[r["record_path"]][factor] = levels[(i + offset) % len(levels)]
    return plan


# ---------------------------------------------------------------- pool worker
_SEED: int | None = None


def init_worker(config: dict) -> None:
    """config: the clean12 render_config plus seed, texture_dir, input_dir, output_dir."""
    global _SEED
    for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
        os.environ[name] = "1"
    R.init_worker(config, config["input_dir"], config["output_dir"])
    set_texture_dir(config["texture_dir"])
    _texture_files()
    _SEED = int(config["seed"])


def process_batch(jobs: list[dict]) -> list[dict]:
    """jobs: record_path, image_path, plan. One kit invocation renders the whole
    batch to grey masters (scratch, deleted afterwards); each master is then
    augmented and written to OUTPUT/image_path (temp file + atomic rename)."""
    scratch = Path(R.CFG["kit_scratch_dir"]) / f"a{os.getpid()}-{int(time.time() * 1000) % 100000000}"
    staged, render = scratch / "staged", scratch / "render"
    rows, todo = {}, {}
    for job in jobs:
        rel = job["record_path"]
        rows[rel] = {"record_path": rel, "image_path": job["image_path"], "status": "", "error": ""}
        try:
            header = R.INPUT / rel
            prows, names, fs, nsamp, _, comments = R.parse_header(header)
            mv = R.load_mv(header, prows, names, nsamp)
            rid = R.record_id_for(rel)
            if rid in todo:
                raise RuntimeError(f"duplicate staged id {rid}")
            R.stage_record(rid, mv, fs, comments, staged)
            todo[rid] = job
        except Exception as exc:  # noqa: BLE001 - one bad record never stops the batch
            rows[rel].update(status="failed", error=f"{type(exc).__name__}: {exc}"[:1000])
    try:
        if todo:
            try:
                R.run_kit(staged, render)
                masters = R.collect_pngs(render, set(todo))
            except Exception as exc:  # noqa: BLE001
                masters = {}
                for job in todo.values():
                    rows[job["record_path"]].update(status="failed",
                                                    error=f"{type(exc).__name__}: {exc}"[:1000])
            for rid, png in masters.items():
                job = todo[rid]
                row = rows[job["record_path"]]
                try:
                    with Image.open(png) as master:
                        if master.size != (R.MASTER_W, R.MASTER_H):
                            raise RuntimeError(f"master size {master.size}")
                        final, meta = augment_master(master, _SEED, job["record_path"], job["plan"])
                    check_plan(meta, job["plan"])
                    out = R.OUTPUT / job["image_path"]
                    out.parent.mkdir(parents=True, exist_ok=True)
                    temp = out.with_name(out.name + f".{os.getpid()}.part")
                    final.save(temp, "PNG", compress_level=R.CFG["png_compress_level"])
                    os.replace(temp, out)
                    row.update(meta, status="success", image_bytes=out.stat().st_size,
                               ink_bin=job["plan"]["ink_bin"], tilt_bin=job["plan"]["tilt_bin"])
                except Exception as exc:  # noqa: BLE001
                    row.update(status="failed", error=f"{type(exc).__name__}: {exc}"[:1000])
    finally:
        shutil.rmtree(scratch, ignore_errors=True)
    return [rows[job["record_path"]] for job in jobs]
'''

for _name, _source in (("ecg_clean_worker.py", CLEAN_WORKER_SOURCE), ("augment_worker.py", AUGMENT_WORKER_SOURCE)):
    _path = WORK_DIR / _name
    compile(_source, str(_path), "exec")
    if not _path.is_file() or _path.read_text() != _source:
        _path.write_text(_source)
WORKER_SHA256 = hashlib.sha256((CLEAN_WORKER_SOURCE + AUGMENT_WORKER_SOURCE).encode()).hexdigest()
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
if str(WORK_DIR) not in os.environ.get("PYTHONPATH", "").split(os.pathsep):
    os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, [str(WORK_DIR), os.environ.get("PYTHONPATH")]))
importlib.invalidate_caches()
import ecg_clean_worker
import augment_worker as AW
ecg_clean_worker = importlib.reload(ecg_clean_worker)
AW = importlib.reload(AW)

render_config = {
    "kit_dir": KIT_DIR, "kit_python": sys.executable, "kit_seed": KIT_SEED,
    "kit_fixed_font": KIT_FIXED_FONT, "kit_render_flags": KIT_RENDER_FLAGS, "kit_env": KIT_ENV,
    "kit_scratch_dir": str(KIT_SCRATCH_DIR), "max_duration_s": MAX_DURATION_S,
    "png_compress_level": PNG_COMPRESS_LEVEL,
}
WORKER_CONFIG = {**render_config, "seed": AUGMENT_SEED, "texture_dir": str(TEXTURE_DIR),
                 "input_dir": str(INPUT_DIR), "output_dir": str(OUTPUT_DIR)}
KIT_SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
print("Workers written; SHA-256", WORKER_SHA256[:16], "| palettes", AW.AUG_PALETTES)
STAGES_DONE.add("workers")

## 7. Balanced plan, run configuration and balance audit

The EC2 notebook's plan cell. `AW.balanced_plan` assigns every record one level of each balanced factor; it is deterministic for a given `AUGMENT_SEED`, so every session reproduces it exactly. It writes into the metadata folder, which goes into the final archive:
- `augmentation_plan.csv`;
- `balance_by_condition.csv` (count of each condition × factor × level);
- `record_manifest.csv` (the clean12 labels, for joins);
- `config.json`.

It stops unless every factor is balanced to ±1 overall and inside every `(site, label set)` stratum. It also defines the pool runner used by the preflight.

In [ ]:
plan = AW.balanced_plan(records, AUGMENT_SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "images").mkdir(exist_ok=True)

run_config = {
    "dataset": "HEEDB clean12, augmented paper/photo sheet (one per record)",
    "source_uri": S3_SOURCE, "source_sha256": expected_sha256, "source_etag": source_identity["etag"],
    "record_count": len(records), "seed": AUGMENT_SEED, "worker_sha256": WORKER_SHA256,
    "kit_bundle_sha256": KIT_BUNDLE_SHA256, "render": {k: v for k, v in render_config.items()
                                                       if k not in ("kit_dir", "kit_python", "kit_scratch_dir")},
    "augmentation": "kit grey master (clean12 settings) -> stage4_augment.augment with the balanced "
                    "plan -> JPEG q85 4:2:0 round trip -> clean12 to_final (1024x768)",
    "balanced_factors": {k: list(v) for k, v in AW.PLAN_FACTORS.items()},
    "ink_bins": AW.INK_BINS, "tilt_bins_deg": AW.TILT_BINS,
    "grid_palettes": {k: AW.GRID_PALETTES[k] for k in AW.AUG_PALETTES},
    "always_on": {"wrinkle_texture_strength": AW.TEXTURE_STRENGTH, "wrinkle_clip": AW.TEXTURE_CLIP,
                  "light_ramp": [0.10, 0.22], "corner_falloff": [0.06, 0.18], "tint_sigma": 0.012,
                  "brightness": [0.92, 1.06], "contrast": [0.94, 1.08], "paper_level": AW.PAPER_LEVEL_TARGET,
                  "sensor_noise_sigma": [1.5, 4.5], "jpeg_quality": AW.JPEG_QUALITY},
    "when_on": {"perspective_mag": AW.KEYSTONE_MAG, "crease_amplitude": AW.CREASE_AMP,
                "crease_clip": AW.CREASE_CLIP, "crease_width": AW.CREASE_WIDTH, "paper_bow": AW.BOW_AMP,
                "edge_shadow_depth": [0.10, 0.26], "blur_radius_px": AW.DEFOCUS_RADIUS},
}
config_hash = hashlib.sha256(json.dumps(run_config, sort_keys=True).encode()).hexdigest()
lock_file = OUTPUT_DIR / ".augment_config_sha256"
if lock_file.is_file() and lock_file.read_text().strip() != config_hash:
    raise RuntimeError("OUTPUT_DIR holds images made with different settings. Use a fresh OUTPUT_DIR "
                       "(and ARCHIVE_BASENAME) or restore the previous settings.")
lock_file.write_text(config_hash + "\n")
(OUTPUT_DIR / "config.json").write_text(json.dumps(run_config, indent=2) + "\n")
shutil.copy2(INPUT_DIR / "manifest.csv", OUTPUT_DIR / "record_manifest.csv")

factors = list(AW.PLAN_FACTORS)
with (OUTPUT_DIR / "augmentation_plan.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["record_path", "image_path", "site", "labels", *factors])
    writer.writeheader()
    for r in records:
        writer.writerow({"record_path": r["record_path"], "image_path": r["image_path"],
                         "site": r["site"], "labels": r["labels"], **plan[r["record_path"]]})

audit_failures = []
for factor, levels in AW.PLAN_FACTORS.items():
    counts = Counter(plan[r["record_path"]][factor] for r in records)
    if set(counts) != set(levels) or max(counts.values()) - min(counts.values()) > 1:
        audit_failures.append(f"{factor}: overall counts {dict(counts)}")
strata = defaultdict(list)
for r in records:
    strata[(r["site"], r["labels"])].append(r["record_path"])
for key, paths in strata.items():
    for factor, levels in AW.PLAN_FACTORS.items():
        counts = Counter(plan[p][factor] for p in paths)
        if max(counts.get(l, 0) for l in levels) - min(counts.get(l, 0) for l in levels) > 1:
            audit_failures.append(f"{factor} in stratum {key}: {dict(counts)}")
if audit_failures:
    raise RuntimeError("Plan is not balanced:\n- " + "\n- ".join(audit_failures[:20]))

audit_rows, worst = [], {}
for condition in CONDITIONS:
    paths = [r["record_path"] for r in records if int(record_by_path[r["record_path"]][condition]) == 1]
    if not paths:
        continue
    for factor, levels in AW.PLAN_FACTORS.items():
        counts = Counter(plan[p][factor] for p in paths)
        for level in levels:
            audit_rows.append({"condition": condition, "records": len(paths), "factor": factor, "level": level,
                               "count": counts.get(level, 0), "share": round(counts.get(level, 0) / len(paths), 4)})
        spread = max(counts.get(l, 0) for l in levels) - min(counts.get(l, 0) for l in levels)
        worst[factor] = max(worst.get(factor, 0.0), spread / (len(paths) / len(levels)))
pd.DataFrame(audit_rows).to_csv(OUTPUT_DIR / "balance_by_condition.csv", index=False)
print(f"Plan: {len(records):,} records, balanced to +-1 overall and in all {len(strata)} (site, label set) strata.")
print("Worst per-condition spread (max-min)/expected:", {f: f"{100 * v:.1f}%" for f, v in worst.items()})
display(pd.DataFrame({f: pd.Series(Counter(str(plan[r["record_path"]][f]) for r in records)) for f in factors})
        .T.fillna("").astype(str))

def make_job(r):
    return {"record_path": r["record_path"], "image_path": r["image_path"], "plan": plan[r["record_path"]]}

def run_pool(subset, config, on_row, label):
    """Render subset in BATCH_SIZE tasks on WORKERS spawned processes; on_row(row) per record."""
    batches = iter([subset[i:i + BATCH_SIZE] for i in range(0, len(subset), BATCH_SIZE)])
    finished_n, last_report, t0 = 0, 0, time.monotonic()
    with ProcessPoolExecutor(max_workers=WORKERS, mp_context=mp.get_context("spawn"),
                             initializer=AW.init_worker, initargs=(config,)) as pool:
        pending = {}

        def submit_next():
            batch = next(batches, None)
            if batch is not None:
                pending[pool.submit(AW.process_batch, [make_job(r) for r in batch])] = batch

        for _ in range(MAX_PENDING_BATCHES):
            submit_next()
        while pending:
            finished, _ = wait(pending, return_when=FIRST_COMPLETED)
            for future in finished:
                batch = pending.pop(future)
                try:
                    rows = future.result()
                except Exception as exc:   # task raised outside its per-record handling
                    rows = [{"record_path": r["record_path"], "image_path": r["image_path"], "status": "failed",
                             "error": f"task failure: {type(exc).__name__}: {exc}"[:1000]} for r in batch]
                for row in rows:
                    on_row(row)
                finished_n += len(rows)
                submit_next()
            if finished_n - last_report >= PROGRESS_EVERY or finished_n == len(subset):
                rate = finished_n / max(time.monotonic() - t0, 1e-9)
                print(f"{label}: {finished_n:,}/{len(subset):,} | {rate:.2f} records/s | "
                      f"ETA {(len(subset) - finished_n) / rate / 3600:.1f} h", flush=True)
                last_report = finished_n
    return time.monotonic() - t0

STAGES_DONE.add("plan")

## 8. Shards and resume state (Drive)

- **Shard k** holds `records[k::N_SHARDS]`, every N-th record in manifest order. Each shard is a representative slice, and the assignment is the same in every session.
- **Layout lock:** `run_config.json` in `<basename>.parts/` records the config hash, the record list and the shard layout. A session with different settings is refused, so one archive never mixes settings.
- **Finished shards:** a shard counts as finished only when its `shard_…json` marker exists. The marker is written last, after Drive's checksum of the fragment and its manifest match the local ones. Finished shards must be 0, 1, 2, … with no gaps, because the archive hash runs through them in order.
- **Fragments:** tar members only, with no end-of-archive blocks, so the fragments concatenate into one tar.
- **Running SHA-256:** Python's `hashlib` objects cannot be saved, so the archive-wide hash uses OpenSSL's `SHA256_*` functions on a raw `SHA256_CTX` buffer. That buffer's bytes go into every marker, and the next session continues from them. The cell first checks this hash against `hashlib`, with a save and restore in the middle.

Shard folders left over from an earlier kernel in this runtime are reused (`progress.csv`), except those already on Drive, which are deleted.

In [ ]:
import base64
import ctypes
import ctypes.util
import io
import re
from collections import deque


class ResumableSHA256:
    # SHA-256 whose state survives the session: OpenSSL SHA256_Init/Update/Final on a
    # raw SHA256_CTX (112 bytes; the buffer has slack), saved as base64.
    CTX_BYTES = 256
    _lib = None

    @classmethod
    def _crypto(cls):
        if cls._lib is None:
            import _hashlib  # noqa: F401  (loads the libcrypto Python itself uses)
            for name in filter(None, (ctypes.util.find_library("crypto"), "libcrypto.so.3", "libcrypto.so")):
                try:
                    lib = ctypes.CDLL(name)
                    lib.SHA256_Init, lib.SHA256_Update, lib.SHA256_Final
                except (OSError, AttributeError):
                    continue
                break
            else:
                raise RuntimeError("OpenSSL libcrypto with SHA256_Init/Update/Final not found")
            lib.SHA256_Init.argtypes = [ctypes.c_void_p]
            lib.SHA256_Update.argtypes = [ctypes.c_void_p, ctypes.c_char_p, ctypes.c_size_t]
            lib.SHA256_Final.argtypes = [ctypes.c_char_p, ctypes.c_void_p]
            for fn in (lib.SHA256_Init, lib.SHA256_Update, lib.SHA256_Final):
                fn.restype = ctypes.c_int
            cls._lib = lib
        return cls._lib

    def __init__(self, state=None):
        lib = self._crypto()
        self._ctx = ctypes.create_string_buffer(self.CTX_BYTES)
        if state is None:
            if lib.SHA256_Init(self._ctx) != 1:
                raise RuntimeError("SHA256_Init failed")
        else:
            raw = base64.b64decode(state)
            if len(raw) != self.CTX_BYTES:
                raise ValueError("saved SHA-256 state has the wrong size")
            ctypes.memmove(self._ctx, raw, self.CTX_BYTES)

    def update(self, data):
        if not isinstance(data, bytes):
            data = bytes(data)
        if self._lib.SHA256_Update(self._ctx, data, len(data)) != 1:
            raise RuntimeError("SHA256_Update failed")

    def state(self):
        return base64.b64encode(self._ctx.raw).decode()

    def hexdigest(self):
        ctx = ctypes.create_string_buffer(self._ctx.raw, self.CTX_BYTES)   # Final destroys its context
        out = ctypes.create_string_buffer(32)
        if self._lib.SHA256_Final(out, ctx) != 1:
            raise RuntimeError("SHA256_Final failed")
        return out.raw.hex()


_probe = os.urandom(3 * 2**20 + 77)
_check = ResumableSHA256()
_check.update(_probe[:1_000_003])
_check = ResumableSHA256(_check.state())
_check.update(_probe[1_000_003:])
if _check.hexdigest() != hashlib.sha256(_probe).hexdigest():
    raise RuntimeError("Resumable SHA-256 disagrees with hashlib; the archive hash cannot be carried across sessions")
del _probe, _check


def tar_header(arcname, size, mtime):
    # One regular-file member header, as tarfile.TarFile.addfile writes it (PAX format).
    info = tarfile.TarInfo(arcname)
    info.size, info.mtime, info.mode, info.type = size, int(mtime), 0o644, tarfile.REGTYPE
    return info.tobuf(tarfile.PAX_FORMAT, "utf-8", "surrogateescape")


archive_name = f"{ARCHIVE_BASENAME}.tar"
ARCHIVE_TOPDIR = OUTPUT_DIR.name
parts_folder_id = drive_folder(dataset_folder_id, ARCHIVE_BASENAME + ".parts")

N_SHARDS = math.ceil(len(records) / SHARD_RECORDS)
shards = [records[k::N_SHARDS] for k in range(N_SHARDS)]
def shard_name(k):
    return f"shard_{k:05d}_of_{N_SHARDS:05d}"
def shard_dir(k):
    return SHARDS_DIR / shard_name(k)

shard_layout = {
    "archive": archive_name, "archive_topdir": ARCHIVE_TOPDIR, "config_sha256": config_hash,
    "records": len(records),
    "records_sha256": hashlib.sha256("\n".join(r["record_path"] for r in records).encode()).hexdigest(),
    "n_shards": N_SHARDS, "shard_records": SHARD_RECORDS,
    "assignment": "shard k = records[k::n_shards], manifest order", "matplotlib": MATPLOTLIB_VERSION,
}
_remote_layout = drive_get_bytes(parts_folder_id, "run_config.json")
if _remote_layout is None:
    drive_put_bytes(parts_folder_id, "run_config.json", (json.dumps(shard_layout, indent=2) + "\n").encode())
elif json.loads(_remote_layout) != shard_layout:
    _old = json.loads(_remote_layout)
    _diff = {k: (_old.get(k), v) for k, v in shard_layout.items() if _old.get(k) != v}
    raise RuntimeError(f"{ARCHIVE_BASENAME}.parts/run_config.json on Drive holds a run with other settings "
                       f"{_diff} (Drive vs now). Restore those settings, or use a new ARCHIVE_BASENAME.")

_marker_name = re.compile(r"^shard_(\d{5})_of_" + f"{N_SHARDS:05d}" + r"\.json$")


def load_markers():
    # Finished shards from Drive, in order; refuses gaps, duplicates and foreign settings.
    found = {}
    for item in drive_children(parts_folder_id):
        match = _marker_name.match(item["name"])
        if match:
            k = int(match.group(1))
            if k in found:
                raise RuntimeError(f"two {item['name']} files in {ARCHIVE_BASENAME}.parts/; delete one")
            found[k] = json.loads(drive_download(item["id"]))
    order = sorted(found)
    if order != list(range(len(order))):
        raise RuntimeError(f"Finished shards on Drive are not 0..n-1 without gaps: {order[:20]}")
    for k, marker in found.items():
        if (marker["shard"] != k or marker["config_sha256"] != config_hash
                or marker["records"] != len(shards[k])):
            raise RuntimeError(f"{shard_name(k)}.json does not belong to this run")
    return [found[k] for k in order]


def shard_progress(k):
    # Records of shard k already rendered in this runtime: success rows whose image exists.
    done, path = {}, shard_dir(k) / "progress.csv"
    if path.is_file():
        with path.open(newline="") as handle:
            for row in csv.DictReader(handle):
                if None in row or None in row.values():          # torn line from a killed kernel
                    continue
                if row["status"] == "success" and (shard_dir(k) / row["image_path"]).is_file():
                    done[row["record_path"]] = row
                elif row["status"] != "success":
                    done.pop(row["record_path"], None)
    return done


def write_fragment(k, rows, running):
    # Pack shard k's sheets into a local tar fragment; hash it as it is written.
    path = shard_dir(k) / "fragment.tarpart"
    sha, md5 = hashlib.sha256(), hashlib.md5()
    with path.open("wb") as out:
        def emit(data):
            out.write(data)
            sha.update(data)
            md5.update(data)
            running.update(data)

        for row in rows:
            source = shard_dir(k) / row["image_path"]
            data = source.read_bytes()
            emit(tar_header(f"{ARCHIVE_TOPDIR}/{row['image_path']}", len(data), source.stat().st_mtime))
            emit(data)
            if len(data) % 512:
                emit(b"\0" * (512 - len(data) % 512))
    return path, sha.hexdigest(), md5.hexdigest()


def upload_fragment(k, rows, render_seconds):
    # Upload shard k's fragment to Drive and verify it, then write its manifest and marker.
    if len(markers) != k:
        raise RuntimeError(f"{shard_name(k)} is out of order: {len(markers)} shards are finished")
    prev = markers[-1] if markers else None
    running = ResumableSHA256(prev["running_sha256_state"]) if prev else ResumableSHA256()
    path, sha_hex, md5_hex = write_fragment(k, rows, running)
    size = path.stat().st_size
    name = shard_name(k) + ".tarpart"
    for stale in drive_children(parts_folder_id, name):   # left by a session that ended before its marker
        drive_delete(stale["id"])
    t0 = time.monotonic()
    with path.open("rb") as source:
        def read(offset, n):
            source.seek(offset)
            return source.read(n)
        meta = drive_resumable_upload(drive_resumable_start(parts_folder_id, name, size), size, read)
    check_drive_copy(meta, size, sha_hex, md5_hex)
    seconds = time.monotonic() - t0

    manifest = io.StringIO()
    writer = csv.DictWriter(manifest, fieldnames=AW.MANIFEST_FIELDS, restval="", extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)
    manifest_body = manifest.getvalue().encode("utf-8")
    drive_put_bytes(parts_folder_id, shard_name(k) + ".manifest.csv", manifest_body)
    marker = {
        "shard": k, "n_shards": N_SHARDS, "config_sha256": config_hash, "records": len(rows),
        "file_id": meta["id"], "fragment_bytes": size, "fragment_sha256": sha_hex, "fragment_md5": md5_hex,
        "image_bytes": sum(int(row["image_bytes"]) for row in rows),
        "manifest_sha256": hashlib.sha256(manifest_body).hexdigest(),
        "running_bytes": (prev["running_bytes"] if prev else 0) + size,
        "running_sha256_state": running.state(),
        "render_seconds": round(render_seconds, 1), "upload_seconds": round(seconds, 1),
        "completed_utc": datetime.now(timezone.utc).isoformat(),
    }
    drive_put_bytes(parts_folder_id, shard_name(k) + ".json", (json.dumps(marker, indent=2) + "\n").encode())
    markers.append(marker)
    return marker


markers = load_markers()
next_shard = len(markers)
for _k in range(next_shard):
    shutil.rmtree(shard_dir(_k), ignore_errors=True)   # already on Drive
print(f"{N_SHARDS} shards of ~{len(shards[0]):,} records (~{len(shards[0]) * EST_IMAGE_MIB / 1024:.1f} GiB each); "
      f"{next_shard} already finished in {DRIVE_DATASET_DIR}/{ARCHIVE_BASENAME}.parts/")
if next_shard < N_SHARDS:
    _local = {k: n for k in range(next_shard, N_SHARDS) if shard_dir(k).is_dir() and (n := len(shard_progress(k)))}
    if _local:
        print("Rendered earlier in this runtime (reused):", {shard_name(k): n for k, n in _local.items()})
STAGES_DONE.add("shards")

## 9. Preflight

Takes about a minute. The run continues only if all of these pass:
- **Drive:** a small file and a two-chunk resumable upload are written to `<basename>.parts/`, checked against Drive's checksums and status replies, then deleted;
- **Drive storage:** enough free space for the fragments still to come (a warning if there is not also room for the final archive next to them, see §11);
- **matplotlib** in the kit's Python is `MATPLOTLIB_VERSION`;
- **disk** for `SHARD_LOOKAHEAD + 2` shards and **RAM** for the workers;
- **one real record:** rendered by the kit in a spawned worker, augmented, and checked to be 1024x768. The sheet goes to a scratch folder, which is deleted afterwards.

In [ ]:
pf_failures, pf_warnings = [], []
def pf_check(name, fn):
    try:
        print(f"  ok    {name}: {fn()}", flush=True)
    except Exception as exc:
        pf_failures.append(f"{name}: {type(exc).__name__}: {exc}")
        print(f"  FAIL  {name}: {type(exc).__name__}: {exc}", flush=True)

if GPU_NAME:
    pf_warnings.append(f"GPU {GPU_NAME} is attached but unused; only the {USABLE_CPUS} vCPUs do the work")

def _pf_drive():
    small = f"preflight {datetime.now(timezone.utc).isoformat()}\n".encode()
    meta = drive_put_bytes(parts_folder_id, ".preflight_small.txt", small)
    try:
        assert drive_download(meta["id"]) == small, "small file did not read back"
    finally:
        drive_delete(meta["id"])
    body = os.urandom(256 * 1024 + 4321)           # one full 256 KiB chunk + a short final chunk
    uri = drive_resumable_start(parts_folder_id, ".preflight_resumable.bin", len(body))
    response = drive_http.put(uri, data=body[:256 * 1024], timeout=(30, 120),
                              headers={"Content-Range": f"bytes 0-{256 * 1024 - 1}/{len(body)}"})
    assert response.status_code == 308, f"first chunk -> {response.status_code}"
    assert drive_resumable_status(uri, len(body)) == (256 * 1024, None), "status after the first chunk"
    meta = drive_resumable_upload(uri, len(body), lambda offset, n: body[offset:offset + n])
    try:
        check_drive_copy(meta, len(body), hashlib.sha256(body).hexdigest(), hashlib.md5(body).hexdigest())
        assert drive_download(meta["id"], 1000, 5000) == body[1000:6000], "ranged read differs"
    finally:
        drive_delete(meta["id"])
    return f"write, resumable upload, status, ranged read, delete OK in {ARCHIVE_BASENAME}.parts/"
pf_check("Google Drive", _pf_drive)

def _pf_quota():
    free = drive_free_bytes()
    if free is None:
        return "unlimited"
    fragments = (N_SHARDS - next_shard) * len(shards[0]) * EST_IMAGE_MIB * 2**20
    archive = len(records) * EST_IMAGE_MIB * 2**20
    assert free >= fragments + 2**30, (f"{free / 2**30:.1f} GiB free on Drive, need {(fragments + 2**30) / 2**30:.1f} "
                                       "GiB for the fragments still to come (+1 GiB margin)")
    if free < fragments + archive + 2**30:
        pf_warnings.append(f"Drive has {free / 2**30:.0f} GiB free: not enough to keep all fragments while the "
                           f"{archive / 2**30:.0f} GiB archive is written, so §11 will delete each fragment once copied")
    return f"{free / 2**30:,.0f} GiB free; ~{fragments / 2**30:.0f} GiB of fragments to come, archive ~{archive / 2**30:.0f} GiB"
pf_check("Drive storage", _pf_quota)

def _pf_matplotlib():
    probe = subprocess.run([sys.executable, "-c", "import matplotlib; print(matplotlib.__version__)"],
                           capture_output=True, text=True, timeout=300)
    version = probe.stdout.strip()
    assert version == MATPLOTLIB_VERSION, (f"kit Python has matplotlib {version or probe.stderr.strip()[-300:]}, "
                                           f"need {MATPLOTLIB_VERSION}; rerun §2")
    return f"matplotlib {version} in {sys.executable}"
pf_check("kit matplotlib", _pf_matplotlib)

def _pf_disk():
    need = (SHARD_LOOKAHEAD + 2) * len(shards[0]) * EST_IMAGE_MIB * 2**20 + 5 * 2**30
    free = shutil.disk_usage(SHARDS_DIR).free
    assert free >= need, f"{free / 2**30:.0f} GiB free, need {need / 2**30:.0f} GiB (lower SHARD_LOOKAHEAD)"
    return f"{free / 2**30:.0f} GiB free, {need / 2**30:.0f} GiB needed"
pf_check("disk space", _pf_disk)

def _pf_ram():
    need = WORKERS * 0.7 + 3   # kit ~0.25 GiB, then augment ~0.6 GiB peak per worker; + parent
    assert RAM_GIB >= need, f"{RAM_GIB:.0f} GiB RAM < {need:.0f} GiB for {WORKERS} workers; lower WORKERS"
    return f"{RAM_GIB:.0f} GiB RAM for {WORKERS} workers"
pf_check("RAM", _pf_ram)

def _pf_one_record():
    probe_dir = WORK_DIR / "preflight"
    shutil.rmtree(probe_dir, ignore_errors=True)
    rows = []
    seconds = run_pool(records[:1], {**WORKER_CONFIG, "output_dir": str(probe_dir)}, rows.append, "preflight")
    row = rows[0]
    assert row["status"] == "success", row["error"]
    with Image.open(probe_dir / row["image_path"]) as image:
        assert image.size == (1024, 768) and image.mode == "RGB", (image.size, image.mode)
    shutil.rmtree(probe_dir)
    return (f"{row['record_path']} -> {row['grid_palette']}, tilt {row['tilt_deg']:+.2f}, creases "
            f"{row['fold_creases']}, {int(row['image_bytes']) / 2**20:.2f} MiB in {seconds:.1f} s incl. spawn + kit start")
pf_check("one record end to end (kit + augmentation, spawned worker)", _pf_one_record)

for warning in pf_warnings:
    print("  WARN ", warning)
if pf_failures:
    raise RuntimeError("PREFLIGHT FAIL; fix and rerun this cell:\n- " + "\n- ".join(pf_failures))
print("PREFLIGHT PASS")
STAGES_DONE.add("preflight")

## 10. Render, augment and upload the shards

One pool of `WORKERS` spawned processes works through the unfinished shards in order, in batches of `BATCH_SIZE` (one kit call per batch). It renders up to `SHARD_LOOKAHEAD` shards ahead.

When the oldest open shard is complete:
1. its effect counts are checked against the plan;
2. its tar fragment is written locally and uploaded to Drive, Drive's checksum is compared with the local one, then the manifest and marker are written;
3. its local files are deleted.

The queued batches keep the workers busy while a shard uploads.

A record that fails is retried `RETRY_PASSES` times. If it still fails, the cell raises with `failures_<shard>.csv`, and that shard is not uploaded.

Progress lines give the measured rate and the time left. If the session ends, reconnect and **Run all**: the run continues at the first unfinished shard.

In [ ]:
def shard_job(k, r):
    # The worker writes SHARDS_DIR/<shard>/images/...; rows are logged with the plain image_path.
    return {"record_path": r["record_path"], "image_path": f"{shard_name(k)}/{r['image_path']}",
            "plan": plan[r["record_path"]]}


class OpenShard:
    def __init__(self, k):
        self.k, self.started = k, time.monotonic()
        folder = shard_dir(k)
        folder.mkdir(parents=True, exist_ok=True)
        for stray in folder.rglob("*.part"):     # temp file of a killed worker
            stray.unlink()
        self.done, self.failed, self.tries = shard_progress(k), {}, Counter()
        self.todo = [r for r in shards[k] if r["record_path"] not in self.done]
        self.open = len(self.todo)
        self.log = (folder / "progress.csv").open("a", newline="")
        self.writer = csv.DictWriter(self.log, fieldnames=AW.MANIFEST_FIELDS, restval="", extrasaction="ignore")
        if self.log.tell() == 0:
            self.writer.writeheader()


def finish_shard(st):
    k = st.k
    st.log.close()
    if st.failed:
        path = WORK_DIR / f"failures_{shard_name(k)}.csv"
        with path.open("w", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=AW.MANIFEST_FIELDS, restval="", extrasaction="ignore")
            writer.writeheader()
            writer.writerows(st.failed.values())
        raise RuntimeError(f"{len(st.failed)} records of {shard_name(k)} failed after {RETRY_PASSES} retry pass(es); "
                           f"see {path}. This shard was not uploaded.")
    rows = [st.done[r["record_path"]] for r in shards[k]]
    missing = [row["image_path"] for row in rows if not (shard_dir(k) / row["image_path"]).is_file()]
    if missing:
        raise RuntimeError(f"{shard_name(k)}: {len(missing)} sheets missing on disk, e.g. {missing[:3]}")
    for factor in AW.PLAN_FACTORS:
        rendered_counts = Counter(str(row[factor]) for row in rows)
        planned_counts = Counter(str(plan[r["record_path"]][factor]) for r in shards[k])
        if rendered_counts != planned_counts:
            raise RuntimeError(f"{shard_name(k)} {factor}: rendered {dict(rendered_counts)} != planned {dict(planned_counts)}")
    marker = upload_fragment(k, rows, time.monotonic() - st.started)
    shutil.rmtree(shard_dir(k))
    print(f"{shard_name(k)} finished: {marker['records']:,} sheets, {marker['fragment_bytes'] / 2**30:.2f} GiB "
          f"uploaded in {marker['upload_seconds'] / 60:.1f} min; {len(markers)}/{N_SHARDS} shards on Drive", flush=True)


def log_rows(st, rows, retry=True):
    # Log one finished batch; a failed record is queued again until RETRY_PASSES is used up.
    global rendered
    for row in rows:
        path = row["record_path"]
        row["image_path"] = record_of[path]["image_path"]
        st.writer.writerow(row)
        if row["status"] == "success":
            st.done[path] = row
            st.open -= 1
            rendered += 1
        elif retry:
            st.tries[path] += 1
            if st.tries[path] <= RETRY_PASSES:
                queue.appendleft((st.k, [record_of[path]]))
            else:
                st.failed[path] = row
                st.open -= 1
    st.log.flush()


if next_shard < N_SHARDS:
    record_of = {r["record_path"]: r for r in records}
    states, queue, pending = {}, deque(), {}
    next_open = next_upload = next_shard
    rendered, last_report, run_t0 = 0, 0, time.monotonic()
    print(f"Rendering shards {next_shard}..{N_SHARDS - 1} with {WORKERS} workers", flush=True)
    pool = ProcessPoolExecutor(max_workers=WORKERS, mp_context=mp.get_context("spawn"), initializer=AW.init_worker,
                               initargs=({**WORKER_CONFIG, "output_dir": str(SHARDS_DIR)},))
    try:
        while next_upload < N_SHARDS:
            while next_open < N_SHARDS and next_open <= next_upload + SHARD_LOOKAHEAD:
                need = 2 * len(shards[next_open]) * EST_IMAGE_MIB * 2**20 + 5 * 2**30
                if shutil.disk_usage(SHARDS_DIR).free < need and next_open > next_upload:
                    break                      # wait for an upload to free space
                st = states[next_open] = OpenShard(next_open)
                for i in range(0, len(st.todo), BATCH_SIZE):
                    queue.append((next_open, st.todo[i:i + BATCH_SIZE]))
                next_open += 1
            while queue and len(pending) < MAX_PENDING_BATCHES:
                k, batch = queue.popleft()
                pending[pool.submit(AW.process_batch, [shard_job(k, r) for r in batch])] = (k, batch)
            if states[next_upload].open == 0:
                finish_shard(states.pop(next_upload))
                next_upload += 1
                continue
            if not pending:
                raise RuntimeError(f"Scheduler stalled: {shard_name(next_upload)} is incomplete with nothing queued")
            finished, _ = wait(pending, return_when=FIRST_COMPLETED)
            for future in finished:
                k, batch = pending.pop(future)
                try:
                    rows = future.result()
                except Exception as exc:   # task raised outside its per-record handling
                    rows = [{"record_path": r["record_path"], "status": "failed",
                             "error": f"task failure: {type(exc).__name__}: {exc}"[:1000]} for r in batch]
                log_rows(states[k], rows)
            if rendered - last_report >= PROGRESS_EVERY:
                last_report = rendered
                rate = rendered / (time.monotonic() - run_t0)
                left = (sum(len(shards[k]) for k in range(next_upload, N_SHARDS))
                        - sum(len(s.done) for s in states.values()))
                print(f"  {rendered:,} sheets this session | {rate:.2f} sheets/s | {left:,} left in "
                      f"{N_SHARDS - next_upload} shards | ETA {left / rate / 3600:.1f} h", flush=True)
    finally:
        pool.shutdown(wait=True, cancel_futures=True)
        for future, (k, _batch) in pending.items():   # batches that finished while stopping
            if k in states and future.done() and not future.cancelled() and future.exception() is None:
                log_rows(states[k], future.result(), retry=False)
        for st in states.values():
            st.log.close()
    print(f"All {N_SHARDS} shards rendered and uploaded; this session: {rendered:,} sheets in "
          f"{(time.monotonic() - run_t0) / 3600:.2f} h", flush=True)
else:
    print(f"All {N_SHARDS} shards are already on Drive.")
if len(markers) != N_SHARDS:
    raise RuntimeError(f"{len(markers)}/{N_SHARDS} shards finished")
STAGES_DONE.add("generate")

## 11. Join the shards into the archive on Drive and verify

1. **Manifest check.** The shard manifests are read back from Drive and joined into `manifest.csv`. The cell checks that every record has exactly one successful sheet (113,358) and that the effect counts equal the plan.
2. **Tail.** The metadata files become tar members, followed by the end-of-archive blocks: `manifest.csv`, `augmentation_plan.csv`, `balance_by_condition.csv`, `effect_counts.json`, `record_manifest.csv`, `config.json`, `summary.json`, `failures.csv`, plus `shards.json`. The tail is saved as `tail.bin` and the upload session as `join.json`, so a new session continues the same join.
3. **Join.** One resumable upload creates `<basename>.tar` in `DRIVE_DATASET_DIR`. The bytes come from the fragments, read back from Drive by byte range, then the tail. After a disconnect, Drive reports how many bytes it holds, and the upload continues from there, in this session or the next. An upload session stays valid for a week.
4. **Verify.** Drive's size and `sha256Checksum` of the archive must equal the local values. The SHA-256 is the running hash after the last shard, continued over the tail. It is written to `<basename>.tar.sha256` and read back.

**Drive space.** If Drive has room for the archive on top of the fragments, the fragments stay until the archive is verified. Otherwise each fragment is deleted as soon as the upload has passed it; the length is then left open, so Drive does not reserve the full size up front. In that mode the join cannot be restarted from scratch, so finish it within the week the upload session lasts.

If the archive is already complete and verified (`compose.json`), the cell does nothing. If a file with the archive's name already sits in the folder and was not made by this run (for example a copy of the EC2 archive), the cell refuses to overwrite it.

In [ ]:
COMPOSE_VERIFIED = False
markers = load_markers()
if len(markers) != N_SHARDS:
    raise RuntimeError(f"{len(markers)}/{N_SHARDS} shards are finished; run §10 first")
sidecar_name = archive_name + ".sha256"

_record = drive_get_bytes(parts_folder_id, "compose.json")
_existing = drive_children(dataset_folder_id, archive_name)
if _record is not None:
    _record = json.loads(_record)
    _meta = drive_meta(_record["file_id"])
    if _meta is None or int(_meta["size"]) != _record["bytes"] or (
            _meta.get("sha256Checksum") and _meta["sha256Checksum"] != _record["archive_sha256"]):
        raise RuntimeError(f"{archive_name} on Drive no longer matches compose.json. Delete the archive and "
                           f"{ARCHIVE_BASENAME}.parts/ to start over.")
    archive_sha256 = _record["archive_sha256"]
    _sidecar_body = f"{archive_sha256}  {archive_name}\n".encode()
    if drive_get_bytes(dataset_folder_id, sidecar_name) != _sidecar_body:
        drive_put_bytes(dataset_folder_id, sidecar_name, _sidecar_body)
    COMPOSE_VERIFIED = True
    print(f"Already complete and verified: {DRIVE_DATASET_DIR}/{archive_name} ({_record['bytes'] / 2**30:.2f} GiB), "
          f"SHA-256 {archive_sha256}")

if not COMPOSE_VERIFIED:
    join = drive_get_bytes(parts_folder_id, "join.json")
    join = json.loads(join) if join is not None else None
    if _existing:
        _owned = False                         # the archive this run's join already completed?
        if join is not None:
            try:
                _done = drive_resumable_status(join["session_uri"], join["total_bytes"], join["keep_fragments"])[1]
                _owned = _done is not None and {f["id"] for f in _existing} == {_done["id"]}
            except DriveSessionExpired:
                pass
        if not _owned:
            raise RuntimeError(f"{archive_name} already exists in {DRIVE_DATASET_DIR} and was not made by this run. "
                               "Move or delete it, or use a new ARCHIVE_BASENAME.")
    for k, marker in enumerate(markers):
        if marker["fragment_bytes"] % 512:
            raise RuntimeError(f"{shard_name(k)} fragment is not a whole number of tar blocks")

    if join is None:
        manifest_rows = []
        for k, marker in enumerate(markers):
            body = drive_get_bytes(parts_folder_id, shard_name(k) + ".manifest.csv")
            if body is None or hashlib.sha256(body).hexdigest() != marker["manifest_sha256"]:
                raise RuntimeError(f"{shard_name(k)}.manifest.csv is missing or differs from its marker")
            rows = list(csv.DictReader(io.StringIO(body.decode("utf-8"))))
            if [row["record_path"] for row in rows] != [r["record_path"] for r in shards[k]]:
                raise RuntimeError(f"{shard_name(k)}.manifest.csv does not list that shard's records")
            manifest_rows.extend(rows)
        by_path = {row["record_path"]: row for row in manifest_rows}
        problems = []
        if len(manifest_rows) != EXPECTED_RECORDS or len(by_path) != EXPECTED_RECORDS:
            problems.append(f"{len(manifest_rows):,} rows / {len(by_path):,} records; expected {EXPECTED_RECORDS:,}")
        if set(by_path) != {r["record_path"] for r in records}:
            problems.append("manifest records differ from the clean12 records")
        if any(row["status"] != "success" for row in manifest_rows):
            problems.append("some manifest rows are not 'success'")
        if any(by_path[r["record_path"]]["image_path"] != r["image_path"] for r in records if r["record_path"] in by_path):
            problems.append("some image paths differ from the record paths")
        effect_counts = {}
        for factor in AW.PLAN_FACTORS:
            rendered_counts = Counter(str(row[factor]) for row in manifest_rows)
            planned_counts = Counter(str(plan[r["record_path"]][factor]) for r in records)
            if rendered_counts != planned_counts:
                problems.append(f"{factor}: rendered {dict(rendered_counts)} != planned {dict(planned_counts)}")
            effect_counts[factor] = dict(sorted(rendered_counts.items()))
        if problems:
            raise RuntimeError("Shards do not add up to the dataset:\n- " + "\n- ".join(problems))

        manifest_rows = [by_path[r["record_path"]] for r in records]
        with (OUTPUT_DIR / "manifest.csv").open("w", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=AW.MANIFEST_FIELDS, restval="", extrasaction="ignore")
            writer.writeheader()
            writer.writerows(manifest_rows)
        with (OUTPUT_DIR / "failures.csv").open("w", newline="") as handle:
            csv.DictWriter(handle, fieldnames=AW.MANIFEST_FIELDS).writeheader()
        (OUTPUT_DIR / "effect_counts.json").write_text(json.dumps(effect_counts, indent=2) + "\n")
        (OUTPUT_DIR / "shards.json").write_text(json.dumps({**shard_layout, "shards": [
            {key: m[key] for key in ("shard", "records", "fragment_bytes", "fragment_sha256", "image_bytes",
                                     "render_seconds", "upload_seconds", "completed_utc")} for m in markers]},
            indent=2) + "\n")
        summary = {
            "completed_utc": datetime.now(timezone.utc).isoformat(),
            "source": S3_SOURCE, "source_sha256": expected_sha256,
            "records": len(records), "images": len(manifest_rows), "failed": 0,
            "image_bytes": sum(int(row["image_bytes"]) for row in manifest_rows),
            "render_seconds": round(sum(m["render_seconds"] for m in markers), 1),
            "config_sha256": config_hash, "built_on": "Google Colab, sharded, saved to Google Drive",
            "shards": N_SHARDS, "effect_counts": effect_counts,
        }
        (OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")

        tail = io.BytesIO()
        for name in ("manifest.csv", "augmentation_plan.csv", "balance_by_condition.csv", "effect_counts.json",
                     "record_manifest.csv", "config.json", "summary.json", "failures.csv", "shards.json"):
            data = (OUTPUT_DIR / name).read_bytes()
            tail.write(tar_header(f"{ARCHIVE_TOPDIR}/{name}", len(data), (OUTPUT_DIR / name).stat().st_mtime))
            tail.write(data)
            tail.write(b"\0" * (-len(data) % 512))
        tail.write(b"\0" * (2 * 512))                           # end-of-archive blocks
        _length = markers[-1]["running_bytes"] + tail.tell()
        tail.write(b"\0" * (-_length % tarfile.RECORDSIZE))     # pad to a whole record, as tarfile does
        tail = tail.getvalue()
        _running = ResumableSHA256(markers[-1]["running_sha256_state"])
        _running.update(tail)
        _tail_meta = drive_put_bytes(parts_folder_id, "tail.bin", tail)
        total_bytes = markers[-1]["running_bytes"] + len(tail)
        _free = drive_free_bytes()
        keep_fragments = _free is None or _free >= total_bytes + 2**30
        join = {"tail_file_id": _tail_meta["id"], "tail_sha256": hashlib.sha256(tail).hexdigest(),
                "total_bytes": total_bytes, "archive_sha256": _running.hexdigest(), "keep_fragments": keep_fragments,
                "session_uri": drive_resumable_start(dataset_folder_id, archive_name,
                                                     total_bytes if keep_fragments else None),
                "started_utc": datetime.now(timezone.utc).isoformat()}
        drive_put_bytes(parts_folder_id, "join.json", (json.dumps(join, indent=2) + "\n").encode())
    else:
        tail = drive_download(join["tail_file_id"])
        if hashlib.sha256(tail).hexdigest() != join["tail_sha256"]:
            raise RuntimeError("tail.bin on Drive differs from join.json")
        print(f"Continuing the join started {join['started_utc']} (upload sessions last a week)")

    total_bytes, archive_sha256, keep_fragments = join["total_bytes"], join["archive_sha256"], join["keep_fragments"]
    segments, _offset = [], 0                 # (start, end, fragment file id or None for the tail)
    for marker in markers:
        segments.append((_offset, _offset + marker["fragment_bytes"], marker["file_id"]))
        _offset += marker["fragment_bytes"]
    segments.append((_offset, _offset + len(tail), None))
    if segments[-1][1] != total_bytes:
        raise RuntimeError("fragment sizes + tail do not add up to the archive size in join.json")

    def read_archive(offset, n):
        out = bytearray()
        for start, end, file_id in segments:
            if end <= offset or start >= offset + n:
                continue
            lo, hi = max(start, offset), min(end, offset + n)
            out += tail[lo - start:hi - start] if file_id is None else drive_download(file_id, lo - start, hi - lo)
        return bytes(out)

    deleted, _reported, t0 = set(), [-1], time.monotonic()
    def on_commit(committed):
        if not keep_fragments:
            for start, end, file_id in segments:
                if file_id and end <= committed and file_id not in deleted:
                    drive_delete(file_id)
                    deleted.add(file_id)
        step = int(20 * committed / total_bytes)
        if step > _reported[0]:
            _reported[0] = step
            print(f"  join {committed / 2**30:,.1f} / {total_bytes / 2**30:,.1f} GiB "
                  f"({(time.monotonic() - t0) / 60:.1f} min)", flush=True)

    print(f"Joining {N_SHARDS} fragments + {len(tail) / 2**20:.1f} MiB tail -> {DRIVE_DATASET_DIR}/{archive_name} "
          f"({total_bytes / 2**30:.2f} GiB; fragments {'kept until verified' if keep_fragments else 'deleted as copied'})",
          flush=True)
    try:
        meta = drive_resumable_upload(join["session_uri"], total_bytes, read_archive, keep_fragments, on_commit)
    except DriveSessionExpired:
        missing = [m["file_id"] for m in markers if drive_meta(m["file_id"]) is None]
        if missing:
            raise RuntimeError(f"The join's upload session expired after {len(missing)} fragments were deleted; "
                               f"delete {ARCHIVE_BASENAME}.parts/ on Drive and start over.") from None
        join["session_uri"] = drive_resumable_start(dataset_folder_id, archive_name,
                                                    total_bytes if keep_fragments else None)
        join["started_utc"] = datetime.now(timezone.utc).isoformat()
        drive_put_bytes(parts_folder_id, "join.json", (json.dumps(join, indent=2) + "\n").encode())
        print("The upload session had expired; started a new one", flush=True)
        meta = drive_resumable_upload(join["session_uri"], total_bytes, read_archive, keep_fragments, on_commit)
    for _ in range(20):                      # Drive can fill in sha256Checksum a little after the upload
        meta = drive_meta(meta["id"]) or meta
        if meta.get("sha256Checksum"):
            break
        time.sleep(30)
    if int(meta["size"]) != total_bytes or (meta.get("sha256Checksum") and meta["sha256Checksum"] != archive_sha256):
        drive_delete(meta["id"])
        raise RuntimeError(f"The archive on Drive ({meta['size']} bytes, sha256 {meta.get('sha256Checksum')}) differs "
                           f"from the local {total_bytes:,} bytes / {archive_sha256}; it was deleted.")
    if not meta.get("sha256Checksum"):
        print("  WARN  Drive reported no sha256Checksum after 10 min; only the size and each fragment's checksum "
              "are verified", flush=True)

    _sidecar_body = f"{archive_sha256}  {archive_name}\n".encode()
    (WORK_DIR / sidecar_name).write_bytes(_sidecar_body)
    drive_put_bytes(dataset_folder_id, sidecar_name, _sidecar_body)
    drive_put_bytes(parts_folder_id, "compose.json", (json.dumps({
        "file_id": meta["id"], "archive_sha256": archive_sha256, "bytes": total_bytes,
        "completed_utc": datetime.now(timezone.utc).isoformat()}, indent=2) + "\n").encode())
    for marker in markers:
        if marker["file_id"] not in deleted:
            drive_delete(marker["file_id"])
    for name in ("tail.bin", "join.json"):
        for item in drive_children(parts_folder_id, name):
            drive_delete(item["id"])
    COMPOSE_VERIFIED = True
    print(f"Archive written and verified in {(time.monotonic() - t0) / 60:.1f} min: {DRIVE_DATASET_DIR}/{archive_name}")
    print(f"SHA-256 {archive_sha256}" + (" (equals Drive's sha256Checksum)" if meta.get("sha256Checksum") else "")
          + f" -> {sidecar_name}")
STAGES_DONE.add("compose")

## 12. Release the Colab runtime

The runtime is released (`google.colab.runtime.unassign()`), so it stops using compute units, only when **all** of these hold:
- `RELEASE_RUNTIME_WHEN_DONE` is on;
- §11 verified the archive on Drive in this kernel;
- every pipeline cell (§2-§11) completed in this kernel since §1 was last run;
- **no cell raised an error** in that time.

The cell waits `RELEASE_GRACE_SECONDS` first; interrupt the kernel to cancel. Releasing deletes the runtime's disk, but everything is on Drive at that point.

In [ ]:
REQUIRED_STAGES = ["install", "access", "download", "kit", "records", "workers", "plan", "shards",
                   "preflight", "generate", "compose"]
missing_stages = [s for s in REQUIRED_STAGES if s not in STAGES_DONE]
blockers = []
if not RELEASE_RUNTIME_WHEN_DONE:
    blockers.append("RELEASE_RUNTIME_WHEN_DONE is False")
if not globals().get("COMPOSE_VERIFIED"):
    blockers.append("the archive was not verified in this kernel")
if missing_stages:
    blockers.append(f"these cells did not complete since §1: {missing_stages}")
if CELL_ERRORS:
    blockers.append(f"{len(CELL_ERRORS)} cell(s) raised an error since §1, e.g. {CELL_ERRORS[0]}")
if not IN_COLAB:
    blockers.append("not running on Colab")
if blockers:
    print("NOT releasing the runtime:\n- " + "\n- ".join(blockers))
else:
    print(f"All {len(REQUIRED_STAGES)} pipeline cells completed without errors; archive verified: "
          f"{DRIVE_DATASET_DIR}/{archive_name}")
    print(f"Releasing this Colab runtime in {RELEASE_GRACE_SECONDS} s; interrupt the kernel to cancel.", flush=True)
    time.sleep(RELEASE_GRACE_SECONDS)
    from google.colab import runtime
    runtime.unassign()

## Notes

**Colab disk at peak** (on top of what Colab itself uses):

| Item | Size |
|---|---|
| Signal archive + extracted records | ~19 GiB |
| Open shards (`SHARD_LOOKAHEAD + 1` = 3 × ~3.6 GiB) + one fragment being uploaded | ~15 GiB |
| **Total** | **~34 GiB** |

**Drive storage at peak:** ~160 GiB of fragments plus the ~160 GiB archive during the join. With less free space the join deletes fragments as it goes, and the peak stays near ~165 GiB. Drive also limits uploads to about 750 GB per account per day; this run uploads about 340 GB in total (fragments, then the archive).

**Time.** Measured on a 4-core laptop: about 4 s per record per core (kit about 2.6 s, augmentation and PNG about 1.4 s). Colab vCPUs are usually hyperthreads, so expect a bit more per vCPU. The total is roughly 113,358 × 4-6 s ÷ vCPUs; for example, about 10-16 h on 12 vCPUs. The progress lines in §10 give the measured rate and ETA. The join in §11 reads and writes ~160 GiB within Google; expect tens of minutes to a few hours, depending on Drive throughput.

**Sessions.** One run can span several sessions. Each new runtime:
- signs in again;
- downloads and extracts the signals again (~5 min);
- reruns the preflight;
- continues at the first unfinished shard, or continues the join.

A lost session costs at most the `SHARD_LOOKAHEAD + 1` shards that were rendering.

**Output equals the EC2 notebook's archive:**
- same file name and top folder `clean_ecg_signal_images_clean12_augmented/`, member paths, metadata files and plan; the same PNG for each record;
- the `.sha256` sidecar is in `sha256sum` format.

Differences:
- tar member order (shard by shard), no directory entries (extractors create folders), and the extra `shards.json`;
- `summary.json` has `render_seconds` instead of `generate_cell_seconds`;
- PNG bytes can differ from an EC2 render only if the Python packages differ (matplotlib is pinned).

The clean12 + augmented training notebook reads it from `new_dataset/` as it is.

**Using the output.**
- `manifest.csv` has one row per record: `record_path`, `image_path` and every effect value.
- Join to `record_manifest.csv` on `record_path` for labels and `patient_key`.
- Check a downloaded copy with `sha256sum -c clean_ecg_signal_images_clean12_augmented.tar.sha256`.

**Starting over.** Delete `<DRIVE_DATASET_DIR>/clean_ecg_signal_images_clean12_augmented.parts/` (and the archive, if any) on Drive, or set a new `ARCHIVE_BASENAME`. Empty the Drive trash if you deleted them in the web interface, because trashed files still count against the quota.

**Changing the design.**
1. Edit `../clean12_augmented_ecg_images_ec2/augment_worker.py` (`AUG_PALETTES`, `INK_BINS`, `TILT_BINS`, `PLAN_FACTORS`).
2. Rebuild both notebooks with `python3 make_notebook.py` in each folder.
3. Use a new `ARCHIVE_BASENAME`: the layout lock refuses to mix settings.